# Reproducing Cambrainha et al. (2025) from dandiset 000713

Paper. Cambrainha, Castro, de Vasconcelos, Carelli & Copelli (2025), "Criticality at Work: Scaling in the Mouse Cortex Enhances Performance", PRX Life, doi:10.1103/w49n-2vz8. The claims file (`claims/cambrainha-2025.yaml`) was written from the arXiv preprint 2410.23508 version 2 (15 Sep 2025), with its Appendix and Supplemental Material; codes M1 to M8 and O1 to O11 below refer to the method notes at its top.

Dataset. DANDI:000713 (Allen Institute, Visual Behavior Neuropixels), version 0.240702.1725 (doi:10.48324/dandi.000713/0.240702.1725). The paper cites the Allen dataset (its ref. 60) without a DANDI version; this is the only published version of the dandiset, and its draft has not changed since (both modified 2024-07-02), so it is pinned as the most recent published version. The files are streamed from the archive; nothing is downloaded.

What is read. Per session file: the units table (peak channel, quality metrics, spike times of V1 units only), the electrodes table (structure of each channel), the stimulus presentation tables (stimulus blocks, image onsets and omitted flashes) and the trials table. The probe LFP files are not read.

| claim | tested or blocked |
|---|---|
| sessions-selected | tested |
| fig3a-kurtosis-broader-than-surrogate | tested |
| fig3c-scaling-hit-rate-higher | tested |
| fig3c-hit-rate-rises-with-threshold | tested |
| fig3d-relative-gain-24 | tested |
| figs1-other-window-lengths | tested |
| figs2-strict-units | tested |
| fig4a-active-most-scaling | tested |
| fig4-exponents-active | tested |
| fig4-exponents-passive | tested |
| fig4-exponents-nni | tested |
| fig4bc-static-exponents-trend | tested |
| figs3-exponents-stable-with-threshold | tested |
| figs4-initial-kurtosis-poor-predictor | tested |
| figs5a-firing-rate-order | tested |
| figs5c-hit-rate-firing-rate-nonmonotonic | tested |
| figs7-fatigue-halves | tested |
| figs8-halves-and-phase-order | tested |

The dataset holds everything the claims need, so every claim was tested. This is the revision after the first verification round: the five claims the first version blocked, because their threshold grid started at a surrogate maximum above the plotted range, are now judged on grids started at every threshold below it (see "How readings decide"), and the sections "The surrogate reference" and "Trial density" and the cells marked "Verifier round 1" add the outputs the verifier asked for. The outcomes are in the summary at the end.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
import time
import zlib
import itertools
import resource
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import criticality as cr

PAPER = "cambrainha-2025"


def seed(claim):
    """The claim's seed (protocol, author step 3)."""
    return zlib.crc32(f"{PAPER}/{claim}".encode())


SEED_OFFSETS = [0, 1000, 2000, 3000, 4000]
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
T_START = time.time()
print("dandiset", cr.DANDISET_ID, "version", cr.DANDISET_VERSION)

## Survey of the dataset

Every session file is opened and its units, electrodes and trials tables are read (metadata and array shapes, not spike times). The V1 units are those whose peak channel's `location` is `VISp`.

In [ ]:
assets = cr.session_assets()
with ThreadPoolExecutor(12) as ex:
    survey = pd.DataFrame(list(ex.map(cr.survey, assets)))
survey = survey.sort_values("path").reset_index(drop=True)
print("session files:", len(survey), " subjects:", survey.subject.nunique())
print("session types:", survey.session_type.value_counts().to_dict())
locs = sorted(set(l for ls in survey.locations for l in ls))
print("structures with 'VIS' in their name:", [l for l in locs if "VIS" in l])
print("duplicated session identifiers:", int(survey.session.duplicated().sum()),
      "; sessions sharing total spike count and unit count:",
      int(survey.duplicated(["total_spikes", "n_units"]).sum()))
print(survey[["n_units", "visp_all", "visp_good", "visp_default", "visp_valid", "n_trials"]]
      .describe().round(1).to_string())

## Processing every session

All sessions with at least 128 V1 units (reading (a) of O1, counting both > 128 and >= 128) are streamed and analysed window by window: binning, the three CVs (O2), the momentum-space kurtosis under both poolings (O3) and before coarse graining (kappa_0), one ISI-shuffled surrogate per window under the claim's seed and under the four seed variants, and the real-space coarse graining of the real and surrogate (claim seed) windows. Active windows of 10 and 60 s, and the active phase with only `good` units or only default-filter units (for sessions keeping at least 128 of them), are analysed too. Timed on four sessions while preparing (40 s with four processes), the whole was estimated at 17 minutes and 1 GB per process.

The surrogate is the one of `fig3a-kurtosis-broader-than-surrogate`, and every claim that uses the surrogate takes it from there; the strict-unit surrogate is seeded by `figs2-strict-units`.

In [ ]:
proc = survey[survey.visp_all >= cr.MIN_UNITS].reset_index(drop=True)
asset_of = {a["path"]: a for a in assets}
strict_of = {r.path: tuple(k for k, c in (("good", r.visp_good), ("default", r.visp_default))
                           if c >= cr.MIN_UNITS) for r in proc.itertuples()}
t0 = time.time()
results = cr.run_sessions([asset_of[p] for p in proc.path],
                          seed("fig3a-kurtosis-broader-than-surrogate"),
                          seed("figs2-strict-units"), lambda a: strict_of[a["path"]])
print(f"processed {len(results)} sessions in {(time.time() - t0) / 60:.1f} min;",
      "failed:", [r["meta"]["path"] for r in results if "error" in r])
print("peak memory of a worker (GB):",
      round(resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1e6, 2))

## Data check

In [ ]:
ok = [r for r in results if "error" not in r]
sessions_ok = ok
W_all = cr.window_table(ok)
sel_gt = set(proc.path[proc.visp_all > cr.MIN_UNITS])
print("sessions processed:", len(ok), "; with > 128 V1 units (the analysis set):", len(sel_gt),
      "from", proc[proc.path.isin(sel_gt)].subject.nunique(), "mice")
print("V1 units per analysed session: min", proc[proc.path.isin(sel_gt)].visp_all.min(),
      "median", proc[proc.path.isin(sel_gt)].visp_all.median(),
      "max", proc[proc.path.isin(sel_gt)].visp_all.max())
print("spike times sorted within every unit:", all(r["spikes_sorted"] for r in ok))

# Stimulus blocks: names, order and durations
blk = []
for r in ok:
    for b, (name, s0, s1, n) in sorted(r["blocks"].items()):
        blk.append(dict(path=r["meta"]["path"], block=b, name=name.split("_presentations")[0],
                        start=s0, dur_min=(s1 - s0) / 60, n=n))
blk = pd.DataFrame(blk)
print("\nstimulus blocks (minutes), over sessions:")
print(blk.groupby(["block", "name"]).dur_min.describe()[["count", "min", "50%", "max"]].round(2)
      .to_string())

# Phases (O4) and windows
dur = pd.DataFrame([dict(path=r["meta"]["path"], **{p: sum(b - a for a, b in s) / 60
                                                    for p, s in r["spans"].items()})
                    for r in ok])
print("\nphase durations (min):")
print(dur.describe().loc[["min", "50%", "max"]].round(2).to_string())
print("sessions with more than 30 min between the active and passive blocks:",
      dur[dur.nni_a > 30][["path", "nni_a"]].round(2).to_dict("records"))
print("\nwindows per phase, length and unit set:")
print(W_all.groupby(["phase", "length", "units"]).size().to_string())

# Trials
tt = []
for r in ok:
    t = r["trials"]
    tt.append(dict(go=int((t["go"] & ~t["aborted"]).sum()), hit=int(t["hit"].sum()),
                   miss=int(t["miss"].sum()), catch=int(t["catch"].sum()),
                   aborted=int(t["aborted"].sum()), auto=int(t["auto_rewarded"].sum()),
                   hit_or_miss_not_go=int(((t["hit"] | t["miss"]) & ~t["go"]).sum()),
                   go_without_change_time=int(np.isnan(t["change_time_no_display_delay"][t["go"]]).sum()),
                   last_trial_end_min=(t["stop_time"].max() - r["spans"]["active"][0][1]) / 60))
tt = pd.DataFrame(tt)
print("\ntrials per session:")
print(tt.describe().loc[["min", "50%", "max"]].round(2).to_string())
print("hits + misses equal the non-aborted go trials in every session:",
      bool(((tt.hit + tt.miss) == tt.go).all()))

# Units and rates
print("\nV1 unit firing rate, spikes per 50-ms bin per unit, averaged over a window: "
      f"min {W_all.rate.min():.3f}, median {W_all.rate.median():.3f}, max {W_all.rate.max():.3f}")
w30 = W_all[(W_all.length == 30) & (W_all.units == "all")]
print("30-s windows keeping fewer than 64 active V1 units:", int((w30.N < 64).sum()), "in",
      w30[w30.N < 64].path.nunique(), "sessions")
print("units silent in a 30-s window (left out of that window): median",
      W_all[W_all.length == 30].n_silent.median(), "max", W_all[W_all.length == 30].n_silent.max(),
      "; units kept per window: min", W_all[W_all.length == 30].N.min())

# Clock check: response of V1 to image onsets, shown against omitted flashes
clk = pd.DataFrame([dict(path=r["meta"]["path"], **r["clock"]) for r in ok])
clk["z"] = (clk.shown_mean - clk.omitted_mean) / np.sqrt(clk.shown_sd ** 2 / clk.n_shown
                                                       + clk.omitted_sd ** 2 / clk.n_omitted)
print("\nclock check (V1 spikes 30-130 ms after an image onset minus the 100 ms before):")
print(f"  shown images: median over sessions {clk.shown_mean.median():.1f} spikes;"
      f" omitted flashes: {clk.omitted_mean.median():.1f};"
      f" sessions where shown exceeds omitted by more than 5 SE: {(clk.z > 5).sum()} of {len(clk)};"
      f" smallest z {clk.z.min():.1f}")

# CV exclusions (O2)
W30 = W_all[(W_all.length == 30) & (W_all.units == "all") & W_all.path.isin(sel_gt)]
for cv in ["cv_a", "cv_b", "cv_c"]:
    print(f"windows with {cv} > 5:",
          W30.groupby("phase")[cv].apply(lambda x: f"{(x > 5).sum()} of {len(x)}").to_dict())

The checks above show:

- 153 session files from 81 mice, no session deposited twice (identifiers unique, and no two sessions share their unit and spike counts). Units are assigned to structures by their peak channel; besides VISp, some channels are labelled only `VIS` (a visual area not named), and those units are not counted as V1.
- Every analysed session has the blocks the paper describes, but in a different order from the Appendix: block 0 is the active task (about 60 min), then a 10-s gray screen, the Gabor block (about 15 min), a gray screen of about 5 min, the full-field flashes (about 5 min), and block 5, the passive replay (about 60 min). The Appendix says the gray screen comes last, after the flashes; the files put it between the Gabors and the flashes. Where the time between the active and passive blocks is longer than the usual 26 min (sessions printed above, up to 80.8 min), the NNI (a) phase holds more windows.
- Hits and misses together are the non-aborted go trials; auto-rewarded trials are flagged apart and are not go trials. An active window holds 0 to 4 go trials with an outcome (median 2), fewer than the "three to five trials" of Appendix 5.
- Image onsets of the active block evoke a V1 response 30 to 130 ms later and omitted flashes do not: the difference is at least 4.0 standard errors in every session, and more than 5 in 96 of 98. The spike times and the stimulus tables therefore share a clock, as far as this check shows. The trials table comes from the same stimulus frames; 43 of its outcome trials fall outside every active window (printed below).

## Common definitions

The analysis set is the sessions with more than 128 V1 units (O1 (a)). Readings named in the claims file are run as combinations: O2 (`cv` a, b, c), O3 (`kappa` a, b), O5 (auto-rewarded trials a, b), O6 (`surr_max` over the phase or over all phases). The first reading listed in the claims file is the main one (O2 (a), O3 (a), O5 (a), O6 (a), O4 (a)).

In [ ]:
W = W_all[(W_all.units == "all") & W_all.path.isin(sel_gt)].copy()
mouse_of = dict(zip(survey.path, survey.subject))
W["mouse"] = W.path.map(mouse_of)
res_of = {r["meta"]["path"]: r for r in ok}
CVS, KAPS, AUTOS, SMAXS = ["a", "b", "c"], ["a", "b"], ["a", "b"], ["a", "b"]
GRID_STEP = 0.05


def kept(df, cv):
    return df[df[f"cv_{cv}"] <= cr.CV_MAX]


def surr_max(cv, kap, smax, s=0, nni="nni_a"):
    """kappa_surr_max (O6): (a) over the active phase's kept windows, (b) over all phases."""
    d = kept(W[W.length == 30], cv)
    phases = ["active"] if smax == "a" else ["active", "passive", nni]
    return d[d.phase.isin(phases)][f"surr{s}_kappa_{kap}"].max()


def grid(lo, hi):
    lo = np.ceil(lo / GRID_STEP - 1e-9) * GRID_STEP
    return np.round(np.arange(lo, hi + 1e-9, GRID_STEP), 4)


# Trials placed in active windows, per session and reading (O5); change time is the main anchor
def add_trials(df, auto="a", anchor="change"):
    h = np.zeros(len(df), int)
    n = np.zeros(len(df), int)
    out_of_window = 0
    for p, idx in df.groupby("path").indices.items():
        times, hit = cr.session_trials(res_of[p], auto, anchor)
        hh, nn = cr.trials_per_window(df.t0.values[idx], df.t1.values[idx], times, hit)
        h[idx], n[idx] = hh, nn
        out_of_window += len(times) - nn.sum()
    d = df.copy()
    d["hits"], d["trials"] = h, n
    return d, out_of_window


ACT = W[(W.phase == "active") & (W.length == 30)].reset_index(drop=True)
for auto in AUTOS:
    d, lost = add_trials(ACT, auto)
    ACT[f"hits_{auto}"], ACT[f"trials_{auto}"] = d.hits, d.trials
    print(f"O5 ({auto}): trials placed in active windows {d.trials.sum()}, hits {d.hits.sum()}, "
          f"trials outside every window {lost}; overall hit rate {d.hits.sum() / d.trials.sum():.3f}")
print("trials per active window: median", np.median(ACT.trials_a), "range", ACT.trials_a.min(), "to",
      ACT.trials_a.max())
print("\nkappa_surr_max under each reading (claim seed):")
print(pd.DataFrame([dict(cv=cv, kappa=k, O6=s, kappa_surr_max=round(surr_max(cv, k, s), 3))
                    for cv in CVS for k in KAPS for s in SMAXS]).to_string(index=False))

## How readings decide

The claims file lists readings of details the paper leaves open (O1 to O11). Following the protocol (reader, step 3; author, step 1.6), two kinds are told apart:

- Readings of which trials, windows or subset a reported number counts (O1, O4, O5, and the two ranges of fig3d-relative-gain-24) decide as the criteria say: the claim is reproduced when the result matches under any of them.
- Readings of a method detail (O2, the CV; O6, kappa_surr_max; O8 to O10) decide only when they all give the same outcome. Where they differ, the reading the paper best supports is adopted as an assumption, and the outcome is marked as resting on it. The adopted readings are O2 (a), the CV of the population count: the Appendix describes the CV exclusion as removing rare artefacts ("such high values are not typically observed in awake mice"), and reading (b) removes 9702 of 11639 active windows (printed in the data check) while (a) removes none and (c) 209. O6 (a) for Figure 3 claims, and (b) for Figures 4A and S8, whose shaded area is one for all phases. O9 (a, a, a) and O10 (a), the first listed.

The adopted reading of O6 (the largest surrogate kurtosis) puts kappa_surr_max at 34.29 in the active phase and 48.34 over all phases, above the plotted ranges, so the criteria's grids are empty. That maximum is set by a few surrogate windows (13 of 28335 above 15) (section "The surrogate reference"). The active maximum (34.29) comes from a window with 182 active units. The all-phase maximum (48.34) comes from a passive window of sub-545994 with 87 active units and 84 silent ones, and the next value (36.50) from a window of the same session with 12 active units; with at least 128 active units the all-phase maximum falls to 34.29. For the claims judged on a grid from kappa_surr_max, the criterion is therefore also applied with the grid started at every threshold from 3 up to the top of the plotted range, and at four named starts: the paper's value of about 7 (M4, read from Figures 3C, 3D and 4A), the 99th and 99.9th percentiles of the active surrogate kurtoses, and the maximum over windows with at least 128 active units. For four of these claims (fig3c-scaling-hit-rate-higher, fig3c-hit-rate-rises-with-threshold, fig4a-active-most-scaling, figs8-halves-and-phase-order) every start gives the same outcome, so the start does not matter. figs3-exponents-stable-with-threshold is run only at the named starts and on a variant grid from 4, and there the starts disagree, so the paper's value of about 7 is adopted as an assumption.

The paper's surrogate reference (lowest kurtoses around 6, kappa_surr_max about 7) is far from the one computed here (minimum 2.80, median 3.64 in the active phase), and the correlation-matrix variant does not recover it either (section "The surrogate reference"). So the kurtosis computed here may differ from the paper's. This is tested directly by fig3a-kurtosis-broader-than-surrogate and stated as a question for the authors. The claims that compare windows above and below a kurtosis threshold are still judged, because the hit rate and phase order results here do not depend on where the threshold or the grid start is put (for example, under CV (a) the scaling hit rate is below the no-scaling hit rate at every threshold from 3 to 15). The exponent stability result (figs3-exponents-stable-with-threshold) does depend on the start: from 7 and from the 99th percentile it is partially reproduced, from the 99.9th percentile reproduced. Their outcomes rest on the kurtosis of Eqs. 2 to 5 as printed being the paper's, and each says so.

The two poolings of O3 give the same number: psi is normalised to unit variance for every unit (Eq. 5), so the pooled <psi^4>/<psi^2>^2 is the mean of the units' fourth moments, which is the mean of their kurtoses. The check below confirms it, and the analyses use reading (a) alone.

In [ ]:
d = W[W.length == 30]
print("largest |kappa_a - kappa_b| over windows:", float(np.abs(d.kappa_a - d.kappa_b).max()),
      "; surrogate:", float(np.abs(d.surr0_kappa_a - d.surr0_kappa_b).max()))

## The surrogate reference

Asked for by the verifier in round 1. Which surrogate windows set kappa_surr_max, how it changes when windows with few active units are left out, and its upper percentiles, which serve below as alternative starts of the threshold grids. Then the kurtosis with each unit's counts z-scored before Eq. 2 (the correlation matrix in place of the covariance), to see whether the paper's lowest surrogate kurtoses of about 6 (Appendix 3) are recovered by it.

In [ ]:
S30 = W[(W.length == 30) & W.phase.isin(["active", "passive", "nni_a"])]
cols = ["path", "phase", "t0", "n_units", "N", "n_silent", "rate", "cv_a", "kappa_a", "surr0_kappa_a"]
print("the ten surrogate windows of largest kurtosis (claim seed, active, passive and NNI (a)):")
top = S30.sort_values("surr0_kappa_a", ascending=False).head(10)[cols].copy()
top["path"] = top.path.str.split("/").str[0]
print(top.round(3).to_string(index=False))
print("surrogate windows above 15:", int((S30.surr0_kappa_a > 15).sum()), "of", len(S30),
      "; in the active phase:", int((S30[S30.phase == "active"].surr0_kappa_a > 15).sum()))
print("Spearman correlation of the surrogate kurtosis with N:",
      round(stats.spearmanr(S30.N, S30.surr0_kappa_a)[0], 3), "; with the firing rate:",
      round(stats.spearmanr(S30.rate, S30.surr0_kappa_a)[0], 3))
rows = []
for s in range(5):
    for ph, phases in [("active", ["active"]), ("all phases", ["active", "passive", "nni_a"])]:
        d = S30[S30.phase.isin(phases)]
        for lab, m in [("all", d.N > 0), ("N >= 64", d.N >= 64), ("N >= 128", d.N >= 128)]:
            x = d[m][f"surr{s}_kappa_a"]
            rows.append(dict(seed_offset=SEED_OFFSETS[s], phases=ph, windows=lab, n=int(m.sum()),
                             max=x.max(), p999=np.percentile(x, 99.9), p99=np.percentile(x, 99),
                             p95=np.percentile(x, 95), median=x.median(), min=x.min()))
SREF = pd.DataFrame(rows)
print("\nsurrogate kurtosis by window set (cv a, no window is excluded under it):")
print(SREF[SREF.seed_offset == 0].round(2).to_string(index=False))
print("\nover the five seeds, active phase, all windows:")
print(SREF[(SREF.phases == "active") & (SREF.windows == "all")].round(2).to_string(index=False))
sa = SREF[(SREF.seed_offset == 0) & (SREF.phases == "active")].set_index("windows")
# Alternative starts of the threshold grids (verifier, round 1): the paper's value read from
# Figures 3C, 3D and 4A (M4), the 99th and 99.9th percentiles of the active surrogate kurtoses,
# and the maximum over active surrogate windows with at least 128 active units.
STARTS = {"paper 7": 7.0, "p99": float(sa.loc["all", "p99"]), "p99.9": float(sa.loc["all", "p999"]),
          "max N>=128": float(sa.loc["N >= 128", "max"])}
print("\nalternative grid starts:", {k: round(v, 2) for k, v in STARTS.items()})

# Correlation-matrix variant (z-scored counts before Eq. 2)
rows = []
for ph in ["active", "passive", "nni_a"]:
    d = S30[S30.phase == ph]
    for lab, col in [("data, covariance (Eq. 2)", "kappa_a"), ("surrogate, covariance (Eq. 2)", "surr0_kappa_a"),
                     ("data, correlation", "kappa_z"), ("surrogate, correlation", "surr0_kappa_z")]:
        x = d[col]
        rows.append(dict(phase=ph, kurtosis=lab, min=x.min(), p5=np.percentile(x, 5), median=x.median(),
                         p95=np.percentile(x, 95), max=x.max(),
                         iqr=np.subtract(*np.percentile(x, [75, 25]))))
print("\nkurtosis at N/16 with covariance and with correlation input:")
print(pd.DataFrame(rows).round(2).to_string(index=False))
da = S30[S30.phase == "active"]
print("active windows whose real correlation-input kurtosis exceeds their surrogate's:",
      f"{np.mean(da.kappa_z > da.surr0_kappa_z):.3f}; covariance input: {np.mean(da.kappa_a > da.surr0_kappa_a):.3f}")
print("Spearman correlation of the two real kurtoses across active windows:",
      round(stats.spearmanr(da.kappa_a, da.kappa_z)[0], 3))

**Surrogate reference.** The largest surrogate kurtosis (34.29, active phase) comes from a window with 182 active units, a rate of 0.170 spikes per bin and a CV (a) of 0.53, so the active maximum is not set by a window with few units: leaving out windows with fewer than 64 or 128 active units leaves it at 34.29. The all-phase maximum (48.34, used for fig4a-active-most-scaling and figs8-halves-and-phase-order under O6 (b)) is set by a passive window of sub-545994 with 87 active units, 84 silent ones, a rate of 0.031 and a CV (a) of 3.89, and the next value (36.50) by a window of the same session with 12 active units; with at least 128 active units the all-phase maximum falls to 34.29. These two windows look like the recording dropouts of the data check. Only 13 of 28335 surrogate windows lie above 15. The 99th percentile of the active surrogate kurtoses is 6.51 (6.51 to 6.75 over the five seeds) and the 99.9th 10.43 (10.43 to 13.38). With z-scored counts (the correlation matrix), the surrogate kurtoses are much higher (active minimum 4.09, 5th percentile 7.10, median 15.88, maximum 524.10) and overlap the real ones (median 22.07): a real window exceeds its own surrogate in 78.0% of active windows, against 99.0% with the covariance. Neither input gives lowest surrogate kurtoses of 6 +/- 0.6 together with a surrogate maximum near 7.

## Trial density

Asked for by the verifier in round 1. Appendix 5 says "within each 30-s window only three to five trials are performed". The trials per active window under O5 (a), and, as variants outside O5, with aborted trials counted as failures and with every trial counted, a hit as success; then the scaling and no-scaling hit rates under each. The last lines repeat the comparison with the correlation-input kurtosis.

In [ ]:
def add_counted(df, reading):
    h = np.zeros(len(df), int)
    n = np.zeros(len(df), int)
    for p, idx in df.groupby("path").indices.items():
        times, hit = cr.session_trials_counted(res_of[p], reading)
        h[idx], n[idx] = cr.trials_per_window(df.t0.values[idx], df.t1.values[idx], times, hit)
    return h, n


g315 = grid(3, 15)
rows, TD = [], {}
print("trials per session (all trials of the trials table): median",
      int(np.median([len(res_of[p]["trials"]["hit"]) for p in sel_gt])), "; active windows per session: median",
      int(ACT.groupby("path").size().median()))
for reading in ["go", "go+aborted", "all"]:
    h, n = add_counted(ACT, reading)
    TD[reading] = (h, n)
    k = ACT.kappa_a.values
    hra, hrb, na, nb = cr.hit_curves(k, h, n, g315)
    v = (na > 0) & (nb > 0)
    at = lambda x, th: float(x[np.isclose(g315, th)][0])
    rows.append(dict(trials=reading, per_window_median=np.median(n), per_window_mean=n.mean(),
                     frac_windows_3_to_5=np.mean((n >= 3) & (n <= 5)), hit_rate=h.sum() / n.sum(),
                     scal_above_noscal_3_to_15=np.mean((hra > hrb)[v]), hr_scaling_8=at(hra, 8),
                     hr_noscaling_8=at(hrb, 8), hr_scaling_12=at(hra, 12), hr_noscaling_12=at(hrb, 12)))
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("windows by number of go trials with an outcome (O5 a):",
      pd.Series(TD["go"][1]).value_counts().sort_index().to_dict())

# The correlation-input kurtosis (variant): hit rates over its own range of thresholds
kz = ACT.kappa_z.values
gz = grid(np.percentile(kz, 5), np.percentile(kz, 95))
hra, hrb, na, nb = cr.hit_curves(kz, ACT.hits_a.values, ACT.trials_a.values, gz)
v = (na > 0) & (nb > 0)
print(f"\ncorrelation-input kurtosis, thresholds {gz[0]} to {gz[-1]} (its 5th to 95th percentile), O5 (a):",
      f"scaling above no scaling at {np.mean((hra > hrb)[v]):.3f} of {int(v.sum())} points;",
      f"at the lowest point {hra[0]:.3f} against {hrb[0]:.3f}, at the highest {hra[v][-1]:.3f} against {hrb[v][-1]:.3f}")

**Trial density.** Under O5 (a) an active window holds a median of 2 go trials with an outcome (mean 1.94), and 31.9% of windows hold three to five. Counting aborted trials as failures gives a median of 5 (mean 5.28; 60.4% of windows with three to five), as Appendix 5 describes, but an overall hit rate of 0.236; counting every trial gives 5 (5.58; 62.1%) and 0.223. Under either variant the scaling hit rate is above the no-scaling one at about half the thresholds from 3 to 15 (54.8% and 53.1%), higher at 8 (0.259 against 0.232 with aborted trials) and lower at 12 (0.184 against 0.237). Whether the paper counts aborted trials is asked of a reader (questions file); the claims are tested with O5 as written. With the correlation-input kurtosis, the scaling hit rate is above the no-scaling one at 16.4% of 805 thresholds over its 5th to 95th percentile.

## sessions-selected

> Of the dataset's sessions, 62 (from 44 mice) have more than 128 recorded V1 units, and these are the sessions analysed.

Criterion: V1 units counted per session under O1 (a) (all units) and (b) (the dataset's default quality filters), with > 128 and >= 128; reproduced when, under one reading, both counts are within tolerance (62 +/- 6.2 sessions, 44 +/- 4.4 mice). The `good` label and the valid-channel flag are shown as variants.

In [ ]:
rows = []
for col, label in [("visp_all", "O1 (a) all V1 units"), ("visp_default", "O1 (b) default quality filters"),
                   ("visp_good", "variant: quality == good"), ("visp_valid", "variant: on valid channels"),
                   ("visp_good_valid", "variant: good and on valid channels")]:
    if col not in survey:
        continue
    for op in [">", ">="]:
        m = survey[col] > cr.MIN_UNITS if op == ">" else survey[col] >= cr.MIN_UNITS
        ns, nm = int(m.sum()), survey[m].subject.nunique()
        rows.append(dict(reading=label, rule=f"{op} 128", sessions=ns, mice=nm,
                         sessions_ok=abs(ns - 62) <= 6.2, mice_ok=abs(nm - 44) <= 4.4))
sel_tab = pd.DataFrame(rows)
print(sel_tab.to_string(index=False))
main_rows = sel_tab[sel_tab.reading.str.startswith("O1")]
print("\nreproduced under a reading of the criterion:",
      bool((main_rows.sessions_ok & main_rows.mice_ok).any()))
print("V1 units per session over all 153: quantiles 10/50/90 %:",
      survey.visp_all.quantile([.1, .5, .9]).tolist(), "; sessions with no V1 unit:",
      int((survey.visp_all == 0).sum()))

**Result: not reproduced.** No reading gives both counts: all V1 units give 97 sessions from 68 mice (> 128), the default quality filters leave a single session, and the `good` label (a variant) gives 68 sessions, within tolerance of 62, but from 54 mice, out of tolerance of 44. The analyses below use the 97 sessions of reading (a).

## fig3a-kurtosis-broader-than-surrogate

> Across windows of all sessions, the kurtosis of the coarse-grained activity spans a broad range that overlaps the surrogate distribution and extends well above it; some windows have kurtosis well above any surrogate value, others look like surrogates. Surrogate kurtoses are lowest around 6.

Criterion: per window kurtosis (M1 to M3, readings O2 and O3), the active phase (all phases pooled as a variant). The lowest surrogate kurtoses, as (a) the minimum or (b) the 5th percentile, within 6 +/- 0.6; the real windows' IQR larger than the surrogate's; some real windows below kappa_surr_max (O6) and a majority above it. Partially reproduced when the spread is larger and some windows exceed kappa_surr_max but the lowest surrogate kurtoses are out of tolerance.

In [ ]:


def fig3a_row(cv, smax, s=0, phases=("active",)):
    d = kept(W[(W.length == 30) & W.phase.isin(phases)], cv)
    real, sur = d.kappa_a.values, d[f"surr{s}_kappa_a"].values
    ksm = surr_max(cv, "a", smax, s)
    iqr = lambda x: np.subtract(*np.percentile(x, [75, 25]))
    return dict(cv=cv, O6=smax, seed_offset=SEED_OFFSETS[s], phases="+".join(phases),
                windows=len(d), surr_min=sur.min(), surr_p5=np.percentile(sur, 5),
                surr_median=np.median(sur), kappa_surr_max=ksm, real_median=np.median(real),
                real_p95=np.percentile(real, 95), real_max=real.max(), iqr_real=iqr(real),
                iqr_surr=iqr(sur), frac_below_ksm=np.mean(real < ksm), frac_above_ksm=np.mean(real > ksm))


f3a = pd.DataFrame([fig3a_row(cv, sm) for cv in CVS for sm in SMAXS])
f3a_all = pd.DataFrame([fig3a_row(cv, sm, phases=("active", "passive", "nni_a"))
                        for cv in CVS for sm in SMAXS])
for t in (f3a, f3a_all):
    t["lowest_min_ok"] = (t.surr_min - 6).abs() <= 0.6
    t["lowest_p5_ok"] = (t.surr_p5 - 6).abs() <= 0.6
    t["spread_larger"] = t.iqr_real > t.iqr_surr
    t["some_below"] = t.frac_below_ksm > 0
    t["majority_above"] = t.frac_above_ksm > 0.5
    t["some_above"] = t.frac_above_ksm > 0
print("active phase:")
print(f3a.round(3).to_string(index=False))
print("\nvariant, all phases pooled:")
print(f3a_all.round(3).to_string(index=False))
full = (f3a.lowest_min_ok | f3a.lowest_p5_ok) & f3a.spread_larger & f3a.some_below & f3a.majority_above
part = f3a.spread_larger & f3a.some_above
print("\nreproduced under some reading:", bool(full.any()), "; partial conditions under some reading:",
      bool(part.any()))
print("\nseed variants (main reading, cv a, O6 a):")
f3a_seeds = pd.DataFrame([fig3a_row("a", "a", s) for s in range(5)])
print(f3a_seeds[["seed_offset", "surr_min", "surr_p5", "kappa_surr_max", "frac_above_ksm"]].round(3)
      .to_string(index=False))
print("\nkappa_surr_max per reading and seed:")
print(pd.DataFrame([dict(cv=cv, O6=sm, **{f"+{SEED_OFFSETS[s]}": round(surr_max(cv, "a", sm, s), 2)
                                         for s in range(5)}) for cv in CVS for sm in SMAXS])
      .to_string(index=False))
print("\nreal and surrogate kurtosis by session (active, cv a): sessions whose largest real window "
      "exceeds their largest surrogate:",
      int((W[(W.length == 30) & (W.phase == "active")].groupby("path")
           .apply(lambda g: g.kappa_a.max() > g.surr0_kappa_a.max())).sum()), "of", len(sel_gt))

fig, ax = plt.subplots(figsize=(6, 3.5))
d = W[(W.length == 30) & (W.phase == "active")]
bins = np.linspace(2, 15, 131)
ax.hist(d.surr0_kappa_a, bins, density=True, alpha=.6, label="surrogate")
ax.hist(d.kappa_a, bins, density=True, histtype="step", color="k", label="data")
ax.set_xlabel("kappa (N/16)")
ax.set_ylabel("density")
ax.legend()
print("windows outside the plotted range 2 to 15: data", int(((d.kappa_a < 2) | (d.kappa_a > 15)).sum()),
      ", surrogate", int(((d.surr0_kappa_a < 2) | (d.surr0_kappa_a > 15)).sum()), "of", len(d))
plt.show()

**Result: partially reproduced**, resting on the CV reading (a), an assumption. Real windows spread more than surrogate windows (interquartile range 1.98 against 0.64, active phase), and some real windows exceed kappa_surr_max (largest real 89.8 against 34.29), but the lowest surrogate kurtoses are 2.80 (minimum) and 3.16 (5th percentile), far from 6 +/- 0.6, and the majority condition fails (a fraction of 0.000 of windows, to three decimals, lies above kappa_surr_max). The median kurtosis is 6.18 for real windows against 3.64 for surrogates, and the largest real window exceeds the largest surrogate in 96 of 97 sessions. Under CV reading (c) no real window exceeds kappa_surr_max (31.9 against 34.29), which gives not reproduced; under (b), partially reproduced. Under the five seeds kappa_surr_max ranges from 24.45 to 35.46 and the outcome under CV (a) stays the same.

## fig3c-scaling-hit-rate-higher

> In the active phase, for any kurtosis threshold above the surrogate values, the hit rate pooled over scaling windows is higher than over no-scaling windows and higher than the trial-shuffled control.

Criterion: hit rates as in M5 (Eqs. 11 and 12), the trial shuffle of M6 (500 repetitions), active windows with trials as in O5, at each kappa_th of the grid of O7 (step 0.05) from kappa_surr_max (O6) to 12.5. Reproduced when, under one reading, the scaling hit rate exceeds both the no-scaling hit rate and the mean trial-shuffled hit rate at every grid point; partially reproduced when it holds at 90% or more of points and every failure is within one bootstrap SE (O8) of the comparator; not reproduced otherwise. A reading whose grid is empty (kappa_surr_max above 12.5) cannot meet the criterion. The SE used for a failure is the bootstrap SE of the scaling hit rate (reading (a) of O8), 10^4 resamples. Trials are placed by change time; the trial start is shown as a variant.

In [ ]:
REPS = 500


def shuffled_hits(df, auto, rng, reps=REPS):
    """Per window, the hits of each trial-shuffle repetition (Appendix 6); reps x windows."""
    out = np.zeros((reps, len(df)))
    for p, idx in df.groupby("path").indices.items():
        times, hit = cr.session_trials(res_of[p], auto)
        H = cr.swap_shuffle(hit, rng, reps)
        o = np.argsort(df.t0.values[idx])
        s0, s1 = df.t0.values[idx][o], df.t1.values[idx][o]
        wi = np.searchsorted(s0, times, side="right") - 1
        inside = (wi >= 0) & (times < s1[np.clip(wi, 0, None)])
        for k in np.where(inside)[0]:
            out[:, idx[o[wi[k]]]] += H[:, k]
    return out


def shuffled_curve(kappa, SH, n, g):
    o = np.argsort(kappa)
    k = kappa[o]
    cut = np.searchsorted(k, g, side="right")
    csh = np.concatenate([np.zeros((SH.shape[0], 1)), np.cumsum(SH[:, o], 1)], 1)
    cn = np.r_[0, np.cumsum(n[o])]
    with np.errstate(invalid="ignore", divide="ignore"):
        r = (csh[:, -1:] - csh[:, cut]) / (cn[-1] - cn[cut])
    return np.nanmean(r, 0), np.nanstd(r, 0)


def fig3c_eval(df, cv, auto, smax, rng_sh, rng_bs, ksm=None, hi=12.5, sh_cache=None):
    d = kept(df, cv).reset_index(drop=True)
    ksm = surr_max(cv, "a", smax) if ksm is None else ksm
    g = grid(ksm, hi)
    out = dict(cv=cv, auto=auto, O6=smax, kappa_surr_max=ksm, grid_points=len(g))
    if len(g) == 0:
        return dict(out, verdict="no grid", frac_pass=np.nan), None
    k, h, n = d.kappa_a.values, d[f"hits_{auto}"].values, d[f"trials_{auto}"].values
    hra, hrb, na, nb = cr.hit_curves(k, h, n, g)
    SH = shuffled_hits(d, auto, rng_sh) if sh_cache is None else sh_cache(d, auto)
    shm, shs = shuffled_curve(k, SH, n, g)
    valid = (na > 0) & (nb > 0)
    pas = valid & (hra > hrb) & (hra > shm)
    frac = pas[valid].mean() if valid.any() else np.nan
    se = cr.boot_curves(k, h, n, g, rng_bs)[0]
    fails = valid & ~pas
    within = np.all((np.maximum(hrb, shm) - hra)[fails] <= se[fails]) if fails.any() else True
    verdict = ("reproduced" if valid.any() and pas[valid].all() else
               "partially_reproduced" if frac >= 0.9 and within else "not_reproduced")
    out.update(points_with_trials=int(valid.sum()), frac_pass=frac, failures_within_SE=bool(within),
               hr_scaling_lo=hra[0], hr_noscaling_lo=hrb[0], hr_shuffled_lo=shm[0],
               hr_scaling_hi=hra[valid][-1] if valid.any() else np.nan,
               hr_noscaling_hi=hrb[valid][-1] if valid.any() else np.nan,
               trials_above_hi=na[valid][-1] if valid.any() else np.nan, verdict=verdict)
    curves = pd.DataFrame(dict(kappa_th=g, hr_scaling=hra, hr_noscaling=hrb, hr_shuffled=shm,
                               hr_shuffled_sd=shs, se_scaling=se, trials_above=na, trials_below=nb))
    return out, curves


S3C = seed("fig3c-scaling-hit-rate-higher")
f3c_rows, f3c_curves = [], {}
for cv in CVS:
    for auto in AUTOS:
        for sm in SMAXS:
            row, cur = fig3c_eval(ACT, cv, auto, sm, np.random.default_rng(S3C),
                                  np.random.default_rng(S3C + 1))
            f3c_rows.append(row)
            f3c_curves[(cv, auto, sm)] = cur
f3c = pd.DataFrame(f3c_rows)
print(f3c.round(3).to_string(index=False))
# The full grid of thresholds (from the lowest kappa), main reading, for the shape of the curves
row, cur_full = fig3c_eval(ACT, "a", "a", "a", np.random.default_rng(S3C), np.random.default_rng(S3C + 1),
                           ksm=3.0, hi=15)
print("\nmain reading (cv a, O5 a) on the full range of thresholds 3 to 15 (not the criterion's grid):")
print(cur_full.iloc[::20].round(3).to_string(index=False))
print("points from 3 to 15 where scaling > no-scaling and > shuffled:",
      f"{row['frac_pass']:.3f} of {row['points_with_trials']}")
# Trial start as anchor (variant)
ACT_start = ACT.copy()
d_, _ = add_trials(ACT, "a", anchor="start")
ACT_start["hits_a"], ACT_start["trials_a"] = d_.hits, d_.trials
rows = []
for cv in CVS:
    rows.append(fig3c_eval(ACT_start, cv, "a", "a", np.random.default_rng(S3C),
                           np.random.default_rng(S3C + 1))[0])
print("\nvariant, trials placed by start time:")
print(pd.DataFrame(rows)[["cv", "O6", "kappa_surr_max", "grid_points", "frac_pass", "verdict"]]
      .round(3).to_string(index=False))

In [ ]:
# Seed variants: the trial shuffle, the bootstrap and kappa_surr_max (surrogate seed variants)
rows = []
for s in range(5):
    for cv in CVS:
        for sm in SMAXS:
            ksm = surr_max(cv, "a", sm, s)
            r, _ = fig3c_eval(ACT, cv, "a", sm, np.random.default_rng(S3C + SEED_OFFSETS[s]),
                              np.random.default_rng(S3C + 1 + SEED_OFFSETS[s]), ksm=ksm)
            rows.append(dict(seed_offset=SEED_OFFSETS[s], **{k: r[k] for k in
                                                             ["cv", "O6", "kappa_surr_max", "grid_points",
                                                              "frac_pass", "verdict"]}))
f3c_seeds = pd.DataFrame(rows)
print(f3c_seeds.round(3).to_string(index=False))
best = f3c_seeds.groupby("seed_offset").verdict.apply(
    lambda v: "reproduced" if (v == "reproduced").any() else
    "partially_reproduced" if (v == "partially_reproduced").any() else "not_reproduced")
print("\nbest verdict over readings, per seed:", best.to_dict())

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
c = f3c_curves[("b", "a", "a")]
for a_, cur, title in [(ax[0], cur_full, "cv a, thresholds 3 to 15"), (ax[1], c, "cv b, criterion grid")]:
    if cur is None:
        continue
    a_.plot(cur.kappa_th, cur.hr_scaling, label="scaling")
    a_.plot(cur.kappa_th, cur.hr_noscaling, label="no scaling")
    a_.plot(cur.kappa_th, cur.hr_shuffled, label="trial shuffled")
    a_.set_xlabel("kappa_th")
    a_.set_ylabel("hit rate")
    a_.set_title(title)
    a_.set_ylim(0, 1)
ax[0].legend()
plt.show()
print("plotted hit rates lie in [0, 1]; points without trials are not drawn")

In [ ]:
# Verifier round 1: the criterion with the grid started below kappa_surr_max, under CV (a) and (c).
# For each start from 3 to 12.45 the criterion is applied to the points from the start to 12.5.
def verdict_from(cur, start, hi=12.5):
    c = cur[(cur.kappa_th >= start - 1e-9) & (cur.kappa_th <= hi + 1e-9)]
    valid = (c.trials_above > 0) & (c.trials_below > 0)
    pas = valid & (c.hr_scaling > c.hr_noscaling) & (c.hr_scaling > c.hr_shuffled)
    if not valid.any():
        return "no grid", np.nan
    frac = pas[valid].mean()
    fails = valid & ~pas
    within = bool(np.all((np.maximum(c.hr_noscaling, c.hr_shuffled) - c.hr_scaling)[fails] <= c.se_scaling[fails]))
    return ("reproduced" if pas[valid].all() else "partially_reproduced" if frac >= 0.9 and within
            else "not_reproduced"), frac


def first(s):
    return s.iloc[0] if len(s) else np.nan


def spans_of(g, vs, target):
    """The starts g at which the verdict is `target`, as runs (lo-hi)."""
    m = np.array([v == target for v in vs])
    out, i = [], 0
    while i < len(m):
        if m[i]:
            j = i
            while j + 1 < len(m) and m[j + 1]:
                j += 1
            out.append(f"{g[i]}-{g[j]}" if j > i else f"{g[i]}")
            i = j + 1
        else:
            i += 1
    return ", ".join(out) if out else "none"


rows, cnt = [], []
for cv in ["a", "c"]:
    for auto in AUTOS:
        _, cur = fig3c_eval(ACT, cv, auto, "a", np.random.default_rng(S3C), np.random.default_rng(S3C + 1),
                            ksm=3.0, hi=12.5)
        for name, st in STARTS.items():
            v, f = verdict_from(cur, st)
            c_ = cur[cur.kappa_th >= st - 1e-9]
            rows.append(dict(cv=cv, auto=auto, start=name, kappa_start=round(st, 2), points=len(c_), frac_pass=f,
                             hr_scaling_start=first(c_.hr_scaling), hr_noscaling_start=first(c_.hr_noscaling),
                             hr_shuffled_start=first(c_.hr_shuffled), se_start=first(c_.se_scaling), verdict=v))
        gs = grid(3, 12.45)
        vs = [verdict_from(cur, st)[0] for st in gs]
        cnt.append(dict(cv=cv, auto=auto, starts=len(vs), **pd.Series(vs).value_counts().to_dict(),
                        reproduced_at=spans_of(gs, vs, "reproduced"),
                        partial_at=spans_of(gs, vs, "partially_reproduced")))
print("grid from each alternative start to 12.5:")
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\nverdicts over every start from 3 to 12.45:")
print(pd.DataFrame(cnt).fillna(0).to_string(index=False))

In [ ]:
# Variant outside O5, asked for by the verifier (trial density): aborted trials counted as
# failures, or every trial counted, under CV (a). Asked of a reader (questions file).
def shuffled_counted(reading):
    def f(d, auto, rng=np.random.default_rng(S3C), reps=REPS):
        out = np.zeros((reps, len(d)))
        for p, idx in d.groupby("path").indices.items():
            times, hit = cr.session_trials_counted(res_of[p], reading)
            H = cr.swap_shuffle(hit, rng, reps)
            o = np.argsort(d.t0.values[idx])
            s0, s1 = d.t0.values[idx][o], d.t1.values[idx][o]
            wi = np.searchsorted(s0, times, side="right") - 1
            inside = (wi >= 0) & (times < s1[np.clip(wi, 0, None)])
            for k_ in np.where(inside)[0]:
                out[:, idx[o[wi[k_]]]] += H[:, k_]
        return out
    return f


rows, cnt, CUR_T = [], [], {}
for reading in ["go+aborted", "all"]:
    A_t = ACT.copy()
    A_t["hits_a"], A_t["trials_a"] = TD[reading]
    _, cur = fig3c_eval(A_t, "a", "a", "a", None, np.random.default_rng(S3C + 1), ksm=3.0, hi=12.5,
                        sh_cache=lambda d, auto, r=reading: shuffled_counted(r)(d, auto))
    CUR_T[reading] = cur
    for name, st in {"O6 (a)": surr_max("a", "a", "a"), **STARTS}.items():
        v, f = verdict_from(cur, st)
        c_ = cur[cur.kappa_th >= st - 1e-9]
        rows.append(dict(trials=reading, start=name, kappa_start=round(st, 2), frac_pass=f,
                         hr_scaling_start=first(c_.hr_scaling), hr_noscaling_start=first(c_.hr_noscaling),
                         hr_shuffled_start=first(c_.hr_shuffled), verdict=v))
    gs = grid(3, 12.45)
    vs = [verdict_from(cur, st)[0] for st in gs]
    cnt.append(dict(trials=reading, starts=len(vs), **pd.Series(vs).value_counts().to_dict(),
                    reproduced_at=spans_of(gs, vs, "reproduced"), partial_at=spans_of(gs, vs, "partially_reproduced")))
    k, h, n = A_t.kappa_a.values, A_t.hits_a.values, A_t.trials_a.values
    gall = grid(np.floor(k.min() * 20) / 20, np.ceil(k.max() * 20) / 20)
    hra, hrb, na, nb = cr.hit_curves(k, h, n, gall)
    gain = (hra - hrb) / hrb * 100
    both = (na > 0) & (nb > 0)
    i = np.where(both)[0][np.nanargmax(gain[both])]
    at = lambda th: float(gain[np.isclose(gall, th)][0])
    m7 = both & (gall >= 7 - 1e-9) & (gall <= 12.5 + 1e-9)
    j = np.where(m7)[0][np.nanargmax(gain[m7])]
    print(f"{reading}: relative gain at 8 {at(8):.2f} %, at 12 {at(12):.2f} %; largest over all thresholds "
          f"{gain[i]:.2f} % at {gall[i]} ({na[i]:.0f} trials above); largest from 7 to 12.5 {gain[j]:.2f} % at {gall[j]}")
    s_ = cur[(cur.kappa_th >= 7 - 1e-9)]
    m = s_.hr_scaling.notna()
    print(f"   scaling hit rate from 7 to 12.5: Spearman {stats.spearmanr(s_.kappa_th[m], s_.hr_scaling[m])[0]:.3f}, "
          f"{s_.hr_scaling.iloc[0]:.3f} at 7 (SE {s_.se_scaling.iloc[0]:.3f}) and {s_.hr_scaling[m].iloc[-1]:.3f} "
          f"at {s_.kappa_th[m].iloc[-1]} (SE {s_.se_scaling[m].iloc[-1]:.3f})")
print("\nfig3c criterion with these trials, from each start to 12.5:")
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\nverdicts over every start from 3 to 12.45:")
print(pd.DataFrame(cnt).fillna(0).to_string(index=False))

**Result: not reproduced**, resting on the CV reading (a), an assumption, and on the kurtosis of Eqs. 2 to 5 as printed. Under CV (a) and (c) the criterion's grid from kappa_surr_max (34.29) is empty; started at any threshold from 3 to 12.45 (190 starts, O5 (a) and (b)), it gives not reproduced at every start. From the paper's value of 7 the scaling hit rate exceeds both comparators at none of 111 points (CV (a), O5 (a)); at 7 it is 0.615 against 0.655 for no scaling and 0.605 trial-shuffled, and over thresholds 3 to 15 it is above both at none of 236 points (0.625 against 0.645 at 8, 0.591 against 0.642 at 12). Counting aborted trials or every trial (variants outside O5, asked of a reader) gives not reproduced at every start too (47.7% and 44.1% of points from 7). Only CV (b), which keeps 1937 of 11639 active windows and which the adopted reading rejects, passes: from 8.55 at 67.5% of 80 points (not reproduced) and from 10.30 (O6 (b)) at all 44 points (reproduced), with 7 trials above 12.5.

## fig3c-hit-rate-rises-with-threshold

> As the threshold rises and only windows of higher kurtosis count as scaling, the hit rate over the scaling windows rises.

Criterion: the scaling hit rate of fig3c-scaling-hit-rate-higher on the grid of O7. Reproduced when the Spearman correlation between kappa_th and the scaling hit rate over the grid from kappa_surr_max to 12.5 is positive and the hit rate at 12.5 exceeds that at kappa_surr_max by more than the sum of their bootstrap SEs (O8). Run under every reading of fig3c-scaling-hit-rate-higher; the bootstrap is this claim's (10^4 resamples).

In [ ]:
S3R = seed("fig3c-hit-rate-rises-with-threshold")


def rises_eval(df, cv, auto, smax, rng, ksm=None, hi=12.5, lo_override=None):
    d = kept(df, cv)
    ksm = surr_max(cv, "a", smax) if ksm is None else ksm
    g = grid(ksm if lo_override is None else lo_override, hi)
    out = dict(cv=cv, auto=auto, O6=smax, kappa_surr_max=ksm, lo=g[0] if len(g) else np.nan,
               grid_points=len(g))
    if len(g) < 2:
        return dict(out, verdict="no grid")
    k, h, n = d.kappa_a.values, d[f"hits_{auto}"].values, d[f"trials_{auto}"].values
    hra, _, na, _ = cr.hit_curves(k, h, n, g)
    se = cr.boot_curves(k, h, n, g, rng)[0]
    v = np.isfinite(hra)
    rho = stats.spearmanr(g[v], hra[v])[0]
    last = np.where(v)[0][-1]
    diff = hra[last] - hra[0]
    ok = (rho > 0) and (diff > se[last] + se[0])
    return dict(out, spearman=rho, hr_lo=hra[0], hr_hi=hra[last], kappa_hi=g[last],
                trials_hi=na[last], se_lo=se[0], se_hi=se[last], diff=diff,
                verdict="reproduced" if ok else "not_reproduced")


rr = pd.DataFrame([rises_eval(ACT, cv, auto, sm, np.random.default_rng(S3R))
                   for cv in CVS for auto in AUTOS for sm in SMAXS])
print(rr.round(3).to_string(index=False))
print("\nvariant (not the criterion): main reading over the thresholds 3 to 12.5")
print(pd.DataFrame([rises_eval(ACT, "a", "a", "a", np.random.default_rng(S3R), lo_override=3.0)])
      .round(3).to_string(index=False))
rows = []
for s in range(5):
    for cv in CVS:
        for sm in SMAXS:
            r = rises_eval(ACT, cv, "a", sm, np.random.default_rng(S3R + SEED_OFFSETS[s]),
                           ksm=surr_max(cv, "a", sm, s))
            rows.append(dict(seed_offset=SEED_OFFSETS[s], cv=cv, O6=sm, grid_points=r["grid_points"],
                             spearman=r.get("spearman"), diff=r.get("diff"), verdict=r["verdict"]))
rr_seeds = pd.DataFrame(rows)
print("\nseed variants (O5 a):")
print(rr_seeds.round(3).to_string(index=False))

In [ ]:
# Verifier round 1: the criterion with the grid started below kappa_surr_max (CV (a) and (c))
rows, cnt = [], []
for cv in ["a", "c"]:
    for auto in AUTOS:
        for name, st in STARTS.items():
            r = rises_eval(ACT, cv, auto, "a", np.random.default_rng(S3R), lo_override=st)
            rows.append(dict(start=name, **{k: r.get(k) for k in ["cv", "auto", "lo", "grid_points", "spearman",
                                                                   "hr_lo", "hr_hi", "se_lo", "se_hi", "diff",
                                                                   "verdict"]}))
        d = kept(ACT, cv)
        k, h, n = d.kappa_a.values, d[f"hits_{auto}"].values, d[f"trials_{auto}"].values
        gf = grid(3, 12.5)
        hra, _, _, _ = cr.hit_curves(k, h, n, gf)
        se = cr.boot_curves(k, h, n, gf, np.random.default_rng(S3R))[0]
        vs = []
        for i in range(len(gf) - 1):
            x, y, s_ = gf[i:], hra[i:], se[i:]
            m = np.isfinite(y)
            rho = stats.spearmanr(x[m], y[m])[0]
            last = np.where(m)[0][-1]
            vs.append("reproduced" if rho > 0 and y[last] - y[0] > s_[last] + s_[0] else "not_reproduced")
        cnt.append(dict(cv=cv, auto=auto, starts=len(vs), **pd.Series(vs).value_counts().to_dict(),
                        reproduced_at=spans_of(gf[:-1], vs, "reproduced")))
print("grid from each alternative start to 12.5:")
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\nverdicts over every start from 3 to 12.45 (one bootstrap draw serves every start):")
print(pd.DataFrame(cnt).fillna(0).to_string(index=False))

**Result: not reproduced**, resting on CV (a), an assumption, and on the kurtosis of Eqs. 2 to 5 as printed. The criterion's grid is empty under CV (a) and (c); started at any threshold from 3 to 12.45, it gives not reproduced at all 190 starts. From the paper's value of 7 (CV (a), O5 (a)) the Spearman correlation is -0.248 and the scaling hit rate goes from 0.615 to 0.587; from 3 it is -0.64 (0.641 to 0.587). Under CV (c) from 7 the correlation is 0.205 but the rise, 0.002, is below the sum of the SEs (0.008 and 0.039). With aborted trials counted (a variant) the scaling hit rate falls from 0.261 at 7 to 0.174 at 12.5 (Spearman -0.998). Only CV (b) gives a rise (Spearman 0.917 from 8.55 and 0.784 from 10.30).

## fig3d-relative-gain-24

> The relative performance gain of scaling over no-scaling windows grows with the threshold and reaches up to 24% plus or minus 3%.

Criterion: relative performance (Eq. 13) on the grid of O7 with the readings of O2 to O6. The maximum is taken (a) over the plotted range, kappa_surr_max to 12.5, or (b) over all thresholds at which both scaling and no-scaling windows hold trials. Reproduced when, under one reading, the maximum is within 24 +/- 3 % and the gain at kappa_th = 12 is larger than at kappa_th = 8. The bootstrap SE of the gain (O8 (c)) is shown, with this claim's seed.

In [ ]:
S3D = seed("fig3d-relative-gain-24")
rows = []
for cv in CVS:
    d = kept(ACT, cv)
    k = d.kappa_a.values
    for auto in AUTOS:
        h, n = d[f"hits_{auto}"].values, d[f"trials_{auto}"].values
        gall = grid(np.floor(k.min() * 20) / 20, np.ceil(k.max() * 20) / 20)
        hra, hrb, na, nb = cr.hit_curves(k, h, n, gall)
        gain = (hra - hrb) / hrb * 100
        both = (na > 0) & (nb > 0)
        g8, g12 = gain[np.isclose(gall, 8)][0], gain[np.isclose(gall, 12)][0]
        for sm in SMAXS:
            ksm = surr_max(cv, "a", sm)
            inr = both & (gall >= ksm - 1e-9) & (gall <= 12.5 + 1e-9)
            for rng_name, m in [("a plotted range", inr), ("b all thresholds", both)]:
                if m.any():
                    i = np.where(m)[0][np.nanargmax(gain[m])]
                    mx, at, nab = gain[i], gall[i], na[i]
                else:
                    mx, at, nab = np.nan, np.nan, np.nan
                ok = np.isfinite(mx) and abs(mx - 24) <= 3 and g12 > g8
                rows.append(dict(cv=cv, auto=auto, O6=sm, range=rng_name, kappa_surr_max=ksm,
                                 max_gain_pct=mx, at_kappa=at, trials_above_at_max=nab,
                                 gain8_pct=g8, gain12_pct=g12, verdict="reproduced" if ok else "not_reproduced"))
f3d = pd.DataFrame(rows)
print(f3d.round(2).to_string(index=False))
d = kept(ACT, "a")
k, h, n = d.kappa_a.values, d.hits_a.values, d.trials_a.values
g = grid(3, 15)
hra, hrb, na, nb = cr.hit_curves(k, h, n, g)
se_g = cr.boot_curves(k, h, n, g, np.random.default_rng(S3D))[2]
gain = (hra - hrb) / hrb * 100
print("\nmain reading (cv a, O5 a): relative gain (%) and its bootstrap SE (%) at thresholds 3 to 15")
print(pd.DataFrame(dict(kappa_th=g, gain_pct=gain, se_pct=se_g * 100, trials_above=na)).iloc[::20]
      .round(2).to_string(index=False))
print("range of the gain over 4 to 15 (%):", np.nanmin(gain[g >= 4]).round(2), "to",
      np.nanmax(gain[g >= 4]).round(2))
rows = []
for s in range(5):
    for cv in CVS:
        for sm in SMAXS:
            ksm = surr_max(cv, "a", sm, s)
            sub = f3d[(f3d.cv == cv) & (f3d.O6 == sm) & (f3d.auto == "a")]
            dd = kept(ACT, cv)
            kk, hh, nn = dd.kappa_a.values, dd.hits_a.values, dd.trials_a.values
            gg = grid(ksm, 12.5)
            if len(gg):
                a_, b_, na_, nb_ = cr.hit_curves(kk, hh, nn, gg)
                gn = (a_ - b_) / b_ * 100
                mx = np.nanmax(np.where((na_ > 0) & (nb_ > 0), gn, np.nan))
            else:
                mx = np.nan
            rows.append(dict(seed_offset=SEED_OFFSETS[s], cv=cv, O6=sm, kappa_surr_max=ksm,
                             max_gain_plotted_range=mx))
print("\nseed variants of kappa_surr_max (reading (a) of the range, O5 a):")
print(pd.DataFrame(rows).round(2).to_string(index=False))
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(g, gain)
ax.fill_between(g, gain - se_g * 100, gain + se_g * 100, alpha=.3)
ax.axhline(0, color="k", lw=.5)
ax.set_xlabel("kappa_th")
ax.set_ylabel("relative gain (%)")
ax.set_ylim(-60, 60)
print("points of the plotted gain outside -60 to 60 %:", int(((gain < -60) | (gain > 60)).sum()))
plt.show()
# Verifier round 1: reading (b) of the maximum, restricted to thresholds with at least 30 trials
# on each side (a variant)
rows = []
for cv in CVS:
    d = kept(ACT, cv)
    k = d.kappa_a.values
    for auto in AUTOS:
        gall = grid(np.floor(k.min() * 20) / 20, np.ceil(k.max() * 20) / 20)
        hra, hrb, na, nb = cr.hit_curves(k, d[f"hits_{auto}"].values, d[f"trials_{auto}"].values, gall)
        gain = (hra - hrb) / hrb * 100
        m = (na >= 30) & (nb >= 30)
        i = np.where(m)[0][np.nanargmax(gain[m])]
        rows.append(dict(cv=cv, auto=auto, max_gain_pct=gain[i], at_kappa=gall[i], trials_above=na[i]))
print("\nlargest gain over thresholds with at least 30 trials on each side:")
print(pd.DataFrame(rows).round(2).to_string(index=False))

**Result: not reproduced.** No reading reaches 24 +/- 3 % with a larger gain at 12 than at 8. Under the adopted CV (a), the plotted range is empty and over all thresholds the largest gain is -1.19 % (at 9.85); the gain is -2.99 % at 8 and -7.91 % at 12. With auto-rewarded trials kept as hits (O5 (b)) the largest gain is 54.89 % at 39.50, resting on 2 trials, and the gain at 12 (-5.63 %) is below that at 8 (-2.21 %). CV (c) gives at most 11.54 %. CV (b) gives 66.14 % at 12.40, with 7 trials above it, out of tolerance, and an infinite gain at 3.55 where the no-scaling hit rate is 0. Over thresholds 4 to 15 the main reading's gain ranges from -26.88 to -1.19 %. Over thresholds with at least 30 trials on each side, the largest gain is -1.19 % under CV (a) (0.85 % with O5 (b)), 10.82 % under CV (c) and 29.41 % under CV (b), at 10.30 with 36 trials above it, outside 24 +/- 3 %. With aborted trials counted (a variant, printed under fig3c-scaling-hit-rate-higher) the gain is 11.72 % at 8 and -22.40 % at 12, at most 15.45 % from 7 to 12.5.

## figs1-other-window-lengths

> With 10-s and 60-s windows the results are similar; hit rate over scaling windows rises with the threshold, and the relative gain departs from zero around kappa of 7.

Criterion: the analysis of fig3c and fig3d repeated with 10-s and 60-s windows, with no surrogate range. For each length the test passes when the scaling hit rate exceeds the no-scaling hit rate at every grid point from kappa_th = 8 to 12 and the relative gain at 12 exceeds that at 8. The onset is the smallest kappa_th above which the relative gain stays more than one bootstrap SE (O8 (c)) above zero, judged within 0.7 of 7. Reproduced when both lengths pass and both onsets are within 0.7 of 7; partially reproduced when both pass with an onset out of range, or when one length passes. The onset is searched over thresholds from 3 (the Gaussian value) to 12, a standard choice; the CV readings of O2 and the trial readings of O5 apply.

In [ ]:
SS1 = seed("figs1-other-window-lengths")
lens = {}
for L in (10.0, 60.0):
    d = W[(W.phase == "active") & (W.length == L)].reset_index(drop=True)
    for auto in AUTOS:
        dd, lost = add_trials(d, auto)
        d[f"hits_{auto}"], d[f"trials_{auto}"] = dd.hits, dd.trials
    lens[L] = d
    print(f"{L:.0f}-s windows: {len(d)}; trials placed {d.trials_a.sum()}; "
          f"windows with CV > 5 under a, b, c: {[(d[f'cv_{c}'] > 5).sum() for c in CVS]}")
rows = []
g812 = grid(8, 12)
gon = grid(3, 12)
for L, d0 in lens.items():
    for cv in CVS:
        d = kept(d0, cv)
        k = d.kappa_a.values
        for auto in AUTOS:
            h, n = d[f"hits_{auto}"].values, d[f"trials_{auto}"].values
            hra, hrb, na, nb = cr.hit_curves(k, h, n, g812)
            gain = (hra - hrb) / hrb
            okp = np.all(hra > hrb)
            ok12 = gain[-1] > gain[0]
            ga, gb, _, _ = cr.hit_curves(k, h, n, gon)
            gg = (ga - gb) / gb
            se = cr.boot_curves(k, h, n, gon, np.random.default_rng(SS1))[2]
            above = gg - se > 0
            # smallest threshold from which every later point stays above
            tail = np.flip(np.cumprod(np.flip(above)).astype(bool))
            onset = gon[np.argmax(tail)] if tail.any() else np.nan
            rows.append(dict(length=L, cv=cv, auto=auto, windows=len(d),
                             frac_scaling_gt_noscaling_8_12=np.mean(hra > hrb), gain8_pct=gain[0] * 100,
                             gain12_pct=gain[-1] * 100, trials_above_12=na[-1], passes=bool(okp and ok12),
                             onset=onset, onset_ok=bool(abs(onset - 7) <= 0.7) if np.isfinite(onset) else False))
s1 = pd.DataFrame(rows)
print(s1.round(3).to_string(index=False))


def s1_verdict(t):
    best = "not_reproduced"
    for cv in CVS:
        for auto in AUTOS:
            x = t[(t.cv == cv) & (t.auto == auto)]
            p = x.passes.values
            o = x.onset_ok.values
            if p.all() and o.all():
                return "reproduced"
            if p.all() or p.any():
                best = "partially_reproduced"
    return best


print("\nverdict over readings (one reading for both lengths):", s1_verdict(s1))
# Seed variants of the bootstrap (onset only)
rows = []
for s in range(5):
    for L, d0 in lens.items():
        d = kept(d0, "a")
        k, h, n = d.kappa_a.values, d.hits_a.values, d.trials_a.values
        ga, gb, _, _ = cr.hit_curves(k, h, n, gon)
        gg = (ga - gb) / gb
        se = cr.boot_curves(k, h, n, gon, np.random.default_rng(SS1 + SEED_OFFSETS[s]))[2]
        tail = np.flip(np.cumprod(np.flip(gg - se > 0)).astype(bool))
        rows.append(dict(seed_offset=SEED_OFFSETS[s], length=L,
                         onset=gon[np.argmax(tail)] if tail.any() else np.nan))
print("\nonset under the seed variants (cv a, O5 a):")
print(pd.DataFrame(rows).pivot(index="seed_offset", columns="length", values="onset").to_string())

**Result: not reproduced.** Neither length passes under any reading. With 10-s windows (CV (a), O5 (a)), the scaling hit rate exceeds the no-scaling one at 50.6 % of the points from 8 to 12, the gain is -0.20 % at 8 and 5.45 % at 12, and the onset is 10.55, outside 7 +/- 0.7. With 60-s windows the scaling hit rate exceeds the no-scaling one at 9.9 % of the points, the gain goes from -1.46 % at 8 to -8.09 % at 12, and there is no onset. The onsets are the same under the five seeds.

## figs2-strict-units

> Using only units passing stricter sorting criteria, 43 sessions keep more than 128 units, and the performance results stay qualitatively the same.

Criterion: units under the strict readings of O1, (i) the dataset's default quality filters and (ii) units labelled good. Reproduced when, under one reading, the session count is within 43 +/- 4.3 and, on those sessions with those units, the test of fig3c-scaling-hit-rate-higher is at least partially reproduced and the relative gain at kappa_th = 12 is positive. Partially reproduced when the result holds but no reading gives a count within tolerance. The strict analysis has its own surrogate (one per window, this claim's seed); kappa_surr_max is taken over its active windows (O6 (a); the other phases were not analysed with strict units).

In [ ]:
SS2 = seed("figs2-strict-units")
rows = []
for uset, col in [("default", "visp_default"), ("good", "visp_good")]:
    for op in (">", ">="):
        m = survey[col] > cr.MIN_UNITS if op == ">" else survey[col] >= cr.MIN_UNITS
        rows.append(dict(units=uset, rule=f"{op} 128", sessions=int(m.sum()),
                         mice=survey[m].subject.nunique(), count_ok=abs(int(m.sum()) - 43) <= 4.3))
s2c = pd.DataFrame(rows)
print(s2c.to_string(index=False))
rows = []
for uset, col in [("default", "visp_default"), ("good", "visp_good")]:
    paths = set(survey.path[survey[col] > cr.MIN_UNITS])
    d = W_all[(W_all.units == uset) & W_all.path.isin(paths) & (W_all.phase == "active")].reset_index(drop=True)
    if len(d) == 0:
        rows.append(dict(units=uset, sessions=0, verdict="no sessions"))
        continue
    for auto in AUTOS:
        dd, _ = add_trials(d, auto)
        d[f"hits_{auto}"], d[f"trials_{auto}"] = dd.hits, dd.trials
    for cv in CVS:
        dk = kept(d, cv)
        ksm = dk.surr0_kappa_a.max()
        r, cur = fig3c_eval(d, cv, "a", "a", np.random.default_rng(SS2), np.random.default_rng(SS2 + 1), ksm=ksm)
        k, h, n = dk.kappa_a.values, dk.hits_a.values, dk.trials_a.values
        a_, b_, na_, nb_ = cr.hit_curves(k, h, n, np.array([12.0]))
        g12 = ((a_ - b_) / b_)[0]
        rows.append(dict(units=uset, sessions=d.path.nunique(), cv=cv, windows=len(dk), kappa_surr_max=ksm,
                         grid_points=r["grid_points"], frac_pass=r["frac_pass"], fig3c_verdict=r["verdict"],
                         gain12_pct=g12 * 100, trials_above_12=na_[0]))
s2 = pd.DataFrame(rows)
print(s2.round(3).to_string(index=False))
# Shape of the curves on good units, from kappa 3, cv a
d = W_all[(W_all.units == "good") & W_all.path.isin(set(survey.path[survey.visp_good > cr.MIN_UNITS]))
          & (W_all.phase == "active")].reset_index(drop=True)
dd, _ = add_trials(d, "a")
g = grid(3, 15)
a_, b_, na_, nb_ = cr.hit_curves(d.kappa_a.values, dd.hits.values, dd.trials.values, g)
print("\ngood units, cv a, relative gain (%) at thresholds 4 to 14 (variant, not the criterion's grid):")
print(pd.DataFrame(dict(kappa_th=g, gain_pct=(a_ - b_) / b_ * 100, trials_above=na_)).iloc[20::40]
      .round(2).to_string(index=False))
rows = []
for s in range(5):
    col = f"surr{s}_kappa_a"
    d = W_all[(W_all.units == "good") & W_all.path.isin(set(survey.path[survey.visp_good > cr.MIN_UNITS]))
              & (W_all.phase == "active")].reset_index(drop=True)
    if col not in d or d[col].isna().all():
        continue
    dd, _ = add_trials(d, "a")
    d["hits_a"], d["trials_a"] = dd.hits, dd.trials
    for cv in CVS:
        dk = kept(d, cv)
        r, _ = fig3c_eval(d, cv, "a", "a", np.random.default_rng(SS2 + SEED_OFFSETS[s]),
                          np.random.default_rng(SS2 + 1 + SEED_OFFSETS[s]), ksm=dk[col].max())
        rows.append(dict(seed_offset=SEED_OFFSETS[s], cv=cv, kappa_surr_max=dk[col].max(),
                         grid_points=r["grid_points"], frac_pass=r["frac_pass"], verdict=r["verdict"]))
print("\nseed variants, good units (strict surrogate, shuffle and bootstrap):")
print(pd.DataFrame(rows).round(3).to_string(index=False))

**Result: not reproduced**, resting on CV (a), an assumption. No strict reading gives 43 +/- 4.3 sessions: the default quality filters leave 1 session and the good label 68 (from 54 mice). On the 68 sessions with good units, the relative gain at kappa_th = 12 is -15.92 % under CV (a) and -13.30 % under (c), so the result does not hold. Under CV (b) the fig3c test is partially reproduced (93.8 % of 96 points) and the gain at 12 is 66.08 %, resting on 4 trials, which would give partially reproduced.

## fig4a-active-most-scaling

> For any threshold, the fraction of windows above it is largest in the active phase, smaller in the passive phase and smallest in the non-natural image phase; most non-natural image windows still have higher kurtosis than surrogates.

Criterion: the complementary cumulative distribution of window kurtosis, pooled over sessions, per phase (O4), on the grid from kappa_surr_max (O6) to 15, at points where any phase is above zero. Reproduced when, under one reading, F_active > F_passive > F_NNI at every such point and F_NNI at kappa_surr_max is above 0.5. Partially reproduced when the order holds at 90% or more of points, or holds everywhere but the NNI fraction is 0.5 or below.

In [ ]:


def ccdf(x, g):
    x = np.sort(x)
    return 1 - np.searchsorted(x, g, side="right") / len(x)


def order_eval(groups, g, strict_first=None):
    """Fractions above each threshold per group; the order holds where each group's fraction
    exceeds the next (>= for pairs listed in `strict_first`)."""
    F = {k: ccdf(v, g) for k, v in groups.items()}
    keys = list(groups)
    anyp = np.any([F[k] > 0 for k in keys], 0)
    ok = np.ones(len(g), bool)
    for a, b in zip(keys[:-1], keys[1:]):
        ok &= (F[a] >= F[b]) if strict_first and (a, b) in strict_first else (F[a] > F[b])
    return F, anyp, ok


rows = []
for cv in CVS:
    d = kept(W[W.length == 30], cv)
    for nni in ["nni_a", "nni_b"]:
        for sm in SMAXS:
            ksm = surr_max(cv, "a", sm, nni=nni) if sm == "b" else surr_max(cv, "a", "a")
            g = grid(ksm, 15)
            if len(g) == 0:
                rows.append(dict(cv=cv, O4=nni, O6=sm, kappa_surr_max=ksm, points=0, verdict="no grid"))
                continue
            groups = {p: d[d.phase == p].kappa_a.values for p in ["active", "passive", nni]}
            F, anyp, ok = order_eval(groups, g)
            frac = ok[anyp].mean() if anyp.any() else np.nan
            fn = F[nni][0]
            v = ("reproduced" if anyp.any() and ok[anyp].all() and fn > 0.5 else
                 "partially_reproduced" if anyp.any() and (frac >= 0.9 or ok[anyp].all()) else "not_reproduced")
            rows.append(dict(cv=cv, O4=nni, O6=sm, kappa_surr_max=ksm, points=int(anyp.sum()), frac_order=frac,
                             F_active_lo=F["active"][0], F_passive_lo=F["passive"][0], F_nni_lo=fn, verdict=v))
f4a = pd.DataFrame(rows)
print(f4a.round(4).to_string(index=False))
g = grid(3, 15)
d = kept(W[W.length == 30], "a")
F = {p: ccdf(d[d.phase == p].kappa_a.values, g) for p in ["active", "passive", "nni_a", "nni_b"]}
print("\nfraction of windows above kappa_th, cv a, thresholds 3 to 15 (not the criterion's grid):")
print(pd.DataFrame(dict(kappa_th=g, **F)).iloc[::20].round(4).to_string(index=False))
print("\nmedian window kurtosis per phase and session, then over sessions (cv a):")
print(d.groupby(["phase", "path"]).kappa_a.median().groupby("phase").describe().round(2).to_string())
pp = d.groupby(["path", "phase"]).kappa_a.median().unstack()
print("sessions with median kurtosis active > passive:", int((pp.active > pp.passive).sum()), "of", len(pp),
      "; active > NNI (a):", int((pp.active > pp.nni_a).sum()), "; passive > NNI (a):",
      int((pp.passive > pp.nni_a).sum()))
rows = []
for s in range(5):
    for cv in CVS:
        for sm in SMAXS:
            ksm = surr_max(cv, "a", sm, s)
            g_ = grid(ksm, 15)
            rows.append(dict(seed_offset=SEED_OFFSETS[s], cv=cv, O6=sm, kappa_surr_max=round(ksm, 2), points=len(g_)))
print("\nseed variants change only the start of the grid:")
print(pd.DataFrame(rows).pivot_table(index=["cv", "O6"], columns="seed_offset", values="kappa_surr_max").to_string())
fig, ax = plt.subplots(figsize=(6, 3.5))
for p, f in F.items():
    ax.plot(g, f, label=p)
ax.set_xlabel("kappa_th")
ax.set_ylabel("fraction of windows above")
ax.legend()
plt.show()

In [ ]:
# Verifier round 1: the criterion with the grid started below kappa_surr_max (CV (a) and (c))
def fig4a_from(groups, nni, st, hi=15):
    g = grid(st, hi)
    if len(g) == 0:
        return dict(points=0, verdict="no grid")
    F, anyp, ok = order_eval(groups, g)
    frac = ok[anyp].mean() if anyp.any() else np.nan
    fn = F[nni][0]
    v = ("reproduced" if anyp.any() and ok[anyp].all() and fn > 0.5 else
         "partially_reproduced" if anyp.any() and (frac >= 0.9 or ok[anyp].all()) else "not_reproduced")
    return dict(points=int(anyp.sum()), frac_order=frac, F_active=F["active"][0], F_passive=F["passive"][0],
                F_nni=fn, verdict=v)


rows, cnt = [], []
for cv in ["a", "c"]:
    d = kept(W[W.length == 30], cv)
    for nni in ["nni_a", "nni_b"]:
        groups = {p: d[d.phase == p].kappa_a.values for p in ["active", "passive", nni]}
        for name, st in STARTS.items():
            rows.append(dict(cv=cv, O4=nni, start=name, kappa_start=round(st, 2), **fig4a_from(groups, nni, st)))
        gs = grid(3, 14.95)
        vs = [fig4a_from(groups, nni, st)["verdict"] for st in gs]
        cnt.append(dict(cv=cv, O4=nni, starts=len(vs), **pd.Series(vs).value_counts().to_dict(),
                        reproduced_at=spans_of(gs, vs, "reproduced"),
                        partial_at=spans_of(gs, vs, "partially_reproduced")))
print("grid from each alternative start to 15:")
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\nverdicts over every start from 3 to 14.95:")
print(pd.DataFrame(cnt).fillna(0).to_string(index=False))

**Result: not reproduced**, resting on CV (a), an assumption, and on the kurtosis of Eqs. 2 to 5 as printed. The criterion's grid from kappa_surr_max is empty under CV (a) and (c); started at any threshold from 3 to 14.95, it gives not reproduced at all 240 starts, under both NNI readings. From the paper's value of 7 the order active > passive > NNI holds at none of 161 points: the fractions above 7 are 0.311 active, 0.146 passive and 0.499 NNI (a), so the NNI phase has the most high-kurtosis windows, not the fewest, while active exceeds passive as the paper says. The median kurtosis of a session is higher in the active phase than in NNI (a) in only 16 of 97 sessions, and higher than in passive in 88. Where the criterion's own grid exists (CV (b)), the order holds at none of the points (0 of 130 and of 94).

## figs8-halves-and-phase-order

> Kurtosis is lower in the second half of the active phase than in the first, yet both halves keep more windows above threshold than the passive and non-natural image phases; the non-natural image phase, though it precedes the passive phase, has the lowest kurtosis.

Criterion: complementary cumulative distributions for active first 30 min, active last 30 min, passive and NNI, on the grid from kappa_surr_max to 15. Reproduced when, at every point where any group is above zero, active first half >= active second half > passive > NNI. Partially reproduced when the order of the second half over passive, or the first half over the second, fails at some points. The first half is the windows ending within 30 min of the active block's start, the second half the windows starting at or after 30 min (a standard choice); O2, O4 and O6 readings are run.

In [ ]:
a0 = {p: res_of[p]["spans"]["active"][0][0] for p in res_of}


def halves(df):
    st = df.path.map(a0)
    return df[df.t1 <= st + 1800 + 1e-6], df[df.t0 >= st + 1800 - 1e-6]


rows = []
for cv in CVS:
    d = kept(W[W.length == 30], cv)
    h1, h2 = halves(d[d.phase == "active"])
    for nni in ["nni_a", "nni_b"]:
        for sm in SMAXS:
            ksm = surr_max(cv, "a", sm, nni=nni) if sm == "b" else surr_max(cv, "a", "a")
            g = grid(ksm, 15)
            groups = {"first": h1.kappa_a.values, "second": h2.kappa_a.values,
                      "passive": d[d.phase == "passive"].kappa_a.values, "nni": d[d.phase == nni].kappa_a.values}
            if len(g) == 0:
                rows.append(dict(cv=cv, O4=nni, O6=sm, kappa_surr_max=ksm, points=0, verdict="no grid"))
                continue
            Fh, anyp, ok = order_eval(groups, g, strict_first={("first", "second")})
            pair = lambda a, b, ge=False: float(np.mean((Fh[a] >= Fh[b] if ge else Fh[a] > Fh[b])[anyp]))
            tail_ok = pair("passive", "nni") == 1.0
            v = ("reproduced" if ok[anyp].all() else
                 "partially_reproduced" if tail_ok else "not_reproduced")
            rows.append(dict(cv=cv, O4=nni, O6=sm, kappa_surr_max=ksm, points=int(anyp.sum()),
                             first_ge_second=pair("first", "second", True), second_gt_passive=pair("second", "passive"),
                             passive_gt_nni=pair("passive", "nni"), verdict=v))
s8 = pd.DataFrame(rows)
print(s8.round(3).to_string(index=False))
d = kept(W[W.length == 30], "a")
h1, h2 = halves(d[d.phase == "active"])
print("\nmedian kurtosis (cv a): first half", round(h1.kappa_a.median(), 3), ", second half",
      round(h2.kappa_a.median(), 3), ", passive", round(d[d.phase == "passive"].kappa_a.median(), 3),
      ", NNI (a)", round(d[d.phase == "nni_a"].kappa_a.median(), 3), ", NNI (b)",
      round(d[d.phase == "nni_b"].kappa_a.median(), 3))
print("Mann-Whitney first against second half: p =", f"{stats.mannwhitneyu(h1.kappa_a, h2.kappa_a).pvalue:.2e}")

In [ ]:
# Verifier round 1: the criterion with the grid started below kappa_surr_max (CV (a) and (c))
def figs8_from(groups, st, hi=15):
    g = grid(st, hi)
    if len(g) == 0:
        return dict(points=0, verdict="no grid")
    Fh, anyp, ok = order_eval(groups, g, strict_first={("first", "second")})
    pair = lambda a, b, ge=False: float(np.mean((Fh[a] >= Fh[b] if ge else Fh[a] > Fh[b])[anyp]))
    v = ("reproduced" if ok[anyp].all() else
         "partially_reproduced" if pair("passive", "nni") == 1.0 else "not_reproduced")
    return dict(points=int(anyp.sum()), first_ge_second=pair("first", "second", True),
                second_gt_passive=pair("second", "passive"), passive_gt_nni=pair("passive", "nni"), verdict=v)


rows, cnt = [], []
for cv in ["a", "c"]:
    d = kept(W[W.length == 30], cv)
    h1, h2 = halves(d[d.phase == "active"])
    for nni in ["nni_a", "nni_b"]:
        groups = {"first": h1.kappa_a.values, "second": h2.kappa_a.values,
                  "passive": d[d.phase == "passive"].kappa_a.values, "nni": d[d.phase == nni].kappa_a.values}
        for name, st in STARTS.items():
            rows.append(dict(cv=cv, O4=nni, start=name, kappa_start=round(st, 2), **figs8_from(groups, st)))
        gs = grid(3, 14.95)
        vs = [figs8_from(groups, st)["verdict"] for st in gs]
        cnt.append(dict(cv=cv, O4=nni, starts=len(vs), **pd.Series(vs).value_counts().to_dict(),
                        reproduced_at=spans_of(gs, vs, "reproduced"),
                        partial_at=spans_of(gs, vs, "partially_reproduced")))
print("grid from each alternative start to 15:")
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\nverdicts over every start from 3 to 14.95:")
print(pd.DataFrame(cnt).fillna(0).to_string(index=False))

**Result: not reproduced**, resting on CV (a), an assumption, and on the kurtosis of Eqs. 2 to 5 as printed. Started at any threshold from 3 to 14.95, the criterion gives not reproduced at all 240 starts. From the paper's value of 7 (CV (a), NNI (a)) the first half is at or above the second at every point and the second half above passive at 98.8% of points, but passive lies above NNI at none. Median kurtosis (CV (a)): first half 6.389, second half 6.026 (Mann-Whitney p = 1.4e-42), passive 5.534, NNI 6.994 (a) and 7.216 (b). The second half is lower than the first, as the paper says, but the NNI phase has the highest kurtosis, not the lowest. Where the criterion's own grid exists (CV (b)), passive lies above NNI at none of the points.

## figs4-initial-kurtosis-poor-predictor

> The kurtosis of the activity before coarse graining is a very poor predictor of the kurtosis after coarse graining.

Criterion: kappa_0 at Ncutoff = N (the z-scored binned counts, pooled as in O3), kappa_final at N/16, per window; active phase (all phases pooled as a variant). Spearman correlation across windows; reproduced when its magnitude is below 0.3, partially reproduced between 0.3 and 0.5.

In [ ]:
rows = []
for cv in CVS:
    for ph, phases in [("active", ["active"]), ("all phases", ["active", "passive", "nni_a"])]:
        d = kept(W[(W.length == 30) & W.phase.isin(phases)], cv)
        rho, p = stats.spearmanr(d.kappa0_a, d.kappa_a)
        rows.append(dict(cv=cv, phases=ph, windows=len(d), spearman=rho, p=p,
                         kappa0_median=d.kappa0_a.median(), kappa0_range=f"{d.kappa0_a.min():.1f} to {d.kappa0_a.max():.1f}",
                         verdict="reproduced" if abs(rho) < 0.3 else "partially_reproduced" if abs(rho) < 0.5
                         else "not_reproduced"))
print(pd.DataFrame(rows).round(4).to_string(index=False))
d = kept(W[(W.length == 30) & (W.phase == "active")], "a")
per = d.groupby("path").apply(lambda x: stats.spearmanr(x.kappa0_a, x.kappa_a)[0])
print("\nwithin-session Spearman (active, cv a): median", round(per.median(), 3), "range", round(per.min(), 3),
      "to", round(per.max(), 3))

**Result: reproduced.** The Spearman correlation between kappa_0 and the kurtosis at N/16 across active windows is 0.178 (CV (a), 11639 windows) and 0.190 with all phases; its magnitude is below 0.3 under every reading. Within sessions it ranges from -0.263 to 0.58 (median 0.136).

## figs5a-firing-rate-order

> Mean firing rate over windows above a kurtosis threshold varies almost monotonically with the threshold within each phase (the legend expects it to decrease), and is ordered active < NNI < passive for any threshold, unlike the kurtosis order of Figure 4A.

Criterion: per window, the mean over units of spike counts per bin; averaged over a mouse's windows above kappa_th, then over mice, on the grid from 5 to 12 at points where every phase has windows. Reproduced when active < NNI < passive at every point and, within each phase, the Spearman correlation of rate against kappa_th is negative with magnitude at least 0.8. Partially reproduced when the order holds at 90% or more of points. The rate is over all V1 units of the session, silent ones included.

In [ ]:
g5 = grid(5, 12)


def rate_curve(d, g):
    k, r, m = d.kappa_a.values, d.rate.values, d.mouse.values
    out = np.full(len(g), np.nan)
    for i, th in enumerate(g):
        s = k > th
        if s.any():
            out[i] = pd.Series(r[s]).groupby(m[s]).mean().mean()
    return out


rows = []
curves5 = {}
for cv in CVS:
    d = kept(W[W.length == 30], cv)
    for nni in ["nni_a", "nni_b"]:
        R = {p: rate_curve(d[d.phase == p], g5) for p in ["active", nni, "passive"]}
        curves5[(cv, nni)] = R
        allp = np.all([np.isfinite(R[p]) for p in R], 0)
        order = (R["active"] < R[nni]) & (R[nni] < R["passive"])
        rhos = {p: stats.spearmanr(g5[np.isfinite(R[p])], R[p][np.isfinite(R[p])])[0] for p in R}
        frac = order[allp].mean() if allp.any() else np.nan
        mono = all(rhos[p] < 0 and abs(rhos[p]) >= 0.8 for p in R)
        v = ("reproduced" if allp.any() and order[allp].all() and mono else
             "partially_reproduced" if frac >= 0.9 else "not_reproduced")
        rows.append(dict(cv=cv, O4=nni, points=int(allp.sum()), frac_order=frac,
                         **{f"rho_{p}": rhos[p] for p in R}, verdict=v))
print(pd.DataFrame(rows).round(3).to_string(index=False))
R = curves5[("a", "nni_a")]
print("\nrate (spikes per bin per unit), cv a, NNI (a):")
print(pd.DataFrame(dict(kappa_th=g5, **R)).iloc[::20].round(4).to_string(index=False))
d = kept(W[W.length == 30], "a")
print("\nall windows, mean rate per phase:", d.groupby("phase").rate.mean().round(4).to_dict())

**Result: reproduced**, resting on CV (a), an assumption. Under CV (a), at all 141 thresholds from 5 to 12 the mouse-averaged rate is ordered active < NNI < passive (0.166, 0.199 and 0.223 spikes per bin per unit at 5; 0.135, 0.183 and 0.193 at 12), and it decreases with the threshold in every phase (Spearman -0.991 active, -0.976 NNI (a), -0.998 NNI (b), -0.991 passive). CV (c) gives the same. Under CV (b) the order holds at 97.2 % (NNI (a)) or 86.5 % (NNI (b)) of points and the active and passive correlations are -0.58 and -0.53.

## figs5c-hit-rate-firing-rate-nonmonotonic

> The hit rate over windows below a firing-rate threshold is not a monotonic function of the threshold.

Criterion: active windows split by their mean firing rate at rho_th on a grid of step 0.0025 from 0.05 to 0.20; hit rate pooled as in M5. Reproduced when the below-threshold hit rate rises by more than one bootstrap SE (O8) between two grid points and falls by more than one SE between two others; not reproduced when it is monotonic within that margin. A change between two points counts when it exceeds the larger of their two SEs (this notebook's reading of "one bootstrap SE"; the smaller is shown as well). Bootstrap: this claim's seed, 10^4 resamples.

In [ ]:
S5C = seed("figs5c-hit-rate-firing-rate-nonmonotonic")
gr = np.round(np.arange(0.05, 0.20 + 1e-9, 0.0025), 4)


def nonmono(hr, se, use=np.maximum):
    v = np.isfinite(hr) & np.isfinite(se)
    hr, se = hr[v], se[v]
    D = hr[None, :] - hr[:, None]          # D[i, j] = hr_j - hr_i
    S = use(se[None, :], se[:, None])
    up = np.triu(D > S, 1).any()
    down = np.triu(-D > S, 1).any()
    return bool(up), bool(down)


def s5c_eval(cv, auto, rng):
    d = kept(ACT, cv)
    r, h, n = d.rate.values, d[f"hits_{auto}"].values, d[f"trials_{auto}"].values
    _, hrb, _, nb = cr.hit_curves(r, h, n, gr)
    se = cr.boot_curves(r, h, n, gr, rng)[1]
    up, down = nonmono(hrb, se)
    up2, down2 = nonmono(hrb, se, np.minimum)
    return dict(cv=cv, auto=auto, windows=len(d), rate_range=f"{r.min():.3f} to {r.max():.3f}",
                windows_below_0_05=int((r <= 0.05).sum()), hr_min=np.nanmin(hrb), hr_max=np.nanmax(hrb),
                rises=up, falls=down, rises_minSE=up2, falls_minSE=down2,
                verdict="reproduced" if up and down else "not_reproduced"), (hrb, se, nb)


rows, s5c_cur = [], {}
for cv in CVS:
    for auto in AUTOS:
        r_, c_ = s5c_eval(cv, auto, np.random.default_rng(S5C))
        rows.append(r_)
        s5c_cur[(cv, auto)] = c_
print(pd.DataFrame(rows).round(3).to_string(index=False))
hrb, se, nb = s5c_cur[("a", "a")]
print("\nbelow-threshold hit rate, cv a, O5 a:")
print(pd.DataFrame(dict(rho_th=gr, hit_rate_below=hrb, se=se, trials_below=nb)).iloc[::6].round(4)
      .to_string(index=False))
print("\nseed variants (cv a, O5 a):",
      [s5c_eval("a", "a", np.random.default_rng(S5C + o))[0]["verdict"] for o in SEED_OFFSETS])

**Result: reproduced.** Under every reading the hit rate below rho_th changes direction by more than one SE. Under CV (a) it rises from 0.601 at 0.05 to 0.751 at 0.11 and falls to 0.654 at 0.20 (SEs 0.037, 0.012 and 0.005). The verdict is the same under the five seeds.

## figs7-fatigue-halves

> In each half of the active phase (first and last 30 min), the scaling hit rate still rises above the no-scaling hit rate as the threshold grows, though the overall hit rate is lower in the second half.

Criterion: the analysis of fig3c, separately for windows in the first and the last 30 min of the active phase. Reproduced when, in each half, the relative gain (Eq. 13) is positive at every grid point from kappa_th = 8 to 12 and larger at 12 than at 8, and the overall hit rate (all windows) of the second half is lower than that of the first. Partially reproduced when the gain holds in one half. Halves as in figs8-halves-and-phase-order; O2 and O5 readings are run.

In [ ]:
rows = []
for cv in CVS:
    h1, h2 = halves(kept(ACT, cv))
    for auto in AUTOS:
        res_h = {}
        for name, d in [("first", h1), ("second", h2)]:
            k, h, n = d.kappa_a.values, d[f"hits_{auto}"].values, d[f"trials_{auto}"].values
            a_, b_, na_, nb_ = cr.hit_curves(k, h, n, g812)
            gain = (a_ - b_) / b_
            res_h[name] = dict(pos=bool(np.all(gain > 0)), frac_pos=float(np.mean(gain > 0)), g8=gain[0] * 100,
                               g12=gain[-1] * 100, overall=h.sum() / n.sum(), trials12=na_[-1])
            res_h[name]["holds"] = res_h[name]["pos"] and gain[-1] > gain[0]
        lower = res_h["second"]["overall"] < res_h["first"]["overall"]
        v = ("reproduced" if res_h["first"]["holds"] and res_h["second"]["holds"] and lower else
             "partially_reproduced" if res_h["first"]["holds"] or res_h["second"]["holds"] else "not_reproduced")
        rows.append(dict(cv=cv, auto=auto, **{f"{h}_{k}": v_ for h in res_h for k, v_ in res_h[h].items()},
                         second_lower=lower, verdict=v))
print(pd.DataFrame(rows).round(3).T.to_string())

**Result: not reproduced**, resting on CV (a), an assumption. Under CV (a) the relative gain is negative at every point from 8 to 12 in the first half (-7.98 % at 8, -11.22 % at 12) and at all but 1.2 % of the points in the second (-6.77 %, -20.32 %); the overall hit rate is lower in the second half (0.548 against 0.749), as the paper says. CV (c) gives the same. Under CV (b), with 4 and 6 trials above 12, the gain is positive throughout both halves (reproduced with O5 (a)).

## Real-space exponents (fig4 claims)

Per window, the real-space quantities (M7) are fitted under every reading of O9: K range (a) to the largest K with two clusters or (b) to N/16; P_silence (a) or (b); mu (a) pooled spectra K >= 16 or (b) K = 64. Session values follow O10: (a) the mean of per-window fits (mu only from windows with r^2 >= 0.85) or (b) one fit to the session's averaged quantities. Phases as in O4; windows kept by the CV readings of O2. The same is done on each window's surrogate (claim seed of fig3a-kurtosis-broader-than-surrogate), for the lighter bars of Figure 4.

In [ ]:
t0 = time.time()
fit_rows = []
for r in sessions_ok:
    if r["meta"]["path"] not in sel_gt:
        continue
    for w in r["windows"]:
        if "rs" not in w:
            continue
        base = dict(path=r["meta"]["path"], phase=w["phase"], t0=w["t0"], kappa_a=w["kappa_a"],
                    cv_a=w["cv_a"], cv_b=w["cv_b"], cv_c=w["cv_c"], N=w["N"])
        fit_rows.append(dict(base, kind="real", **cr.window_fits(w["rs"], w["N"])))
        fit_rows.append(dict(base, kind="surrogate", **cr.window_fits(w["surr_rs"], w["N"])))
FITS = pd.DataFrame(fit_rows)
print(f"per-window fits: {len(FITS)} in {time.time() - t0:.0f} s")
cols = [c for c in FITS if c.split("_")[0] in ("alpha", "beta", "z", "mu") and "r2" not in c]
print("windows with a failed fit (NaN), per quantity, real windows:")
print(FITS[FITS.kind == "real"][cols].isna().sum().to_dict())
print("mu windows with r^2 >= 0.85 (real): reading a",
      f"{(FITS[FITS.kind == 'real'].mu_r2_a >= cr.MU_R2_MIN).mean():.3f},",
      "reading b", f"{(FITS[FITS.kind == 'real'].mu_r2_b >= cr.MU_R2_MIN).mean():.3f}")
rs_of = {}
for r in sessions_ok:
    for w in r["windows"]:
        if "rs" in w:
            rs_of[(r["meta"]["path"], w["phase"], w["t0"])] = (w["rs"], w["surr_rs"], w["N"])

EXP = ["alpha", "beta", "mu", "z"]
READ9 = cr.EXP_READINGS


def session_values(phase, cv, reading, agg, kind="real", kmin=None):
    """Session exponents for one phase under one reading (O9) and aggregation (O10)."""
    d = FITS[(FITS.phase == phase) & (FITS.kind == kind) & (FITS[f"cv_{cv}"] <= cr.CV_MAX)]
    if kmin is not None:
        d = d[d.kappa_a > kmin]
    c = cr.reading_columns(*reading)
    out = []
    for p, x in d.groupby("path"):
        if agg == "a":
            mu = x[c["mu"]][x[c["mu_r2"]] >= cr.MU_R2_MIN]
            out.append(dict(path=p, alpha=x[c["alpha"]].mean(), beta=x[c["beta"]].mean(),
                            z=x[c["z"]].mean(), mu=mu.mean() if len(mu) else np.nan, windows=len(x)))
        else:
            rr_ = [rs_of[(p, phase, t)] for t in x.t0]
            rss = [a if kind == "real" else b for a, b, _ in rr_]
            v = cr.session_exponents(rss, [n for _, _, n in rr_], reading, "b")
            out.append(dict(path=p, alpha=v["alpha"], beta=v["beta"], z=v["z"], mu=v["mu"], windows=len(x)))
    return pd.DataFrame(out).set_index("path")


SV = {}
for cv in ["a", "b", "c"]:
    for reading in READ9:
        for agg in "ab":
            for phase in ["active", "passive", "nni_a", "nni_b"]:
                SV[(phase, cv, reading, agg)] = session_values(phase, cv, reading, agg)
print("session values computed:", len(SV), f"({time.time() - t0:.0f} s)")
EXPECTED = {"active": dict(alpha=(1.41, .05), beta=(.77, .04), mu=(.73, .08), z=(.3, .1)),
            "passive": dict(alpha=(1.37, .04), beta=(.78, .04), mu=(.61, .05), z=(.36, .06)),
            "nni": dict(alpha=(1.28, .03), beta=(.82, .03), mu=(.38, .04), z=(.4, .1))}


def means_table(phase, key):
    rows = []
    for cv in CVS:
        for reading in READ9:
            for agg in "ab":
                s = SV[(phase, cv, reading, agg)]
                row = dict(cv=cv, O9="".join(reading), O10=agg, sessions=len(s), sessions_mu=int(s.mu.notna().sum()))
                within = 0
                for e in EXP:
                    m, sd = s[e].mean(), s[e].std()
                    row[f"{e}_mean"], row[f"{e}_sd"] = m, sd
                    ok_ = abs(m - EXPECTED[key][e][0]) <= EXPECTED[key][e][1]
                    row[f"{e}_ok"] = bool(ok_)
                    within += ok_
                row["within"] = within
                rows.append(row)
    return pd.DataFrame(rows)

## fig4-exponents-active

> In the active phase the real-space scaling exponents are alpha 1.41 +/- 0.05, beta 0.77 +/- 0.04, mu 0.73 +/- 0.08 and z 0.3 +/- 0.1 (mean and SD across sessions).

Criterion: exponents as in M7 with the readings of O9 and O10, on active windows (O2). Reproduced when, under one reading, all four means are within tolerance; partially reproduced when three are. Main reading: O9 (a, a, a), O10 (a), O2 (a).

In [ ]:
MA = means_table("active", "active")
show = ["cv", "O9", "O10", "sessions", "sessions_mu"] + [f"{e}_mean" for e in EXP] + [f"{e}_sd" for e in EXP] + ["within"]
print(MA[show].round(3).to_string(index=False))
print("\nlargest number of means within tolerance under a reading:", MA.within.max(), "; readings with 3 or more:",
      int((MA.within >= 3).sum()), "of", len(MA))
print("per exponent, readings within tolerance:", {e: int(MA[f"{e}_ok"].sum()) for e in EXP})
s = SV[("active", "a", ("a", "a", "a"), "a")]
print("\nmain reading, session values (active):")
print(s[EXP].describe().round(3).to_string())
su = session_values("active", "a", ("a", "a", "a"), "a", kind="surrogate")
print("\nsurrogate, main reading:", {e: f"{su[e].mean():.3f} +/- {su[e].std():.3f}" for e in EXP})

In [ ]:
# Verifier round 1: mu fitted over the upper ranks only (ranks 1 to K/2), variants (c) of O9 (a)
# and (d) of O9 (b); not readings of the claims file.
R = FITS[FITS.kind == "real"]
print("mu windows with r^2 >= 0.85 (real):", {v: f"{(R[f'mu_r2_{v}'] >= cr.MU_R2_MIN).mean():.3f}" for v in "abcd"})
print("median r^2:", {v: round(float(R[f"mu_r2_{v}"].median()), 3) for v in "abcd"})
rows = []
for ph, key in [("active", "active"), ("passive", "passive"), ("nni_a", "nni"), ("nni_b", "nni")]:
    for cv in ["a", "c"]:
        for kr in "ab":
            for fs in "ab":
                for mv in "cd":
                    s = session_values(ph, cv, (kr, fs, mv), "a")
                    row = dict(phase=ph, cv=cv, O9=kr + fs + mv, sessions_mu=int(s.mu.notna().sum()))
                    within = 0
                    for e in EXP:
                        m = s[e].mean()
                        row[f"{e}_mean"] = m
                        within += abs(m - EXPECTED[key][e][0]) <= EXPECTED[key][e][1]
                    row["mu_sd"] = s.mu.std()
                    row["within"] = within
                    rows.append(row)
MUV = pd.DataFrame(rows)
print(MUV.round(3).to_string(index=False))
print("\nlargest number of means within tolerance per phase with these variants:",
      MUV.groupby("phase").within.max().to_dict())

**Result: not reproduced.** No reading puts more than two of the four means within tolerance (none of 48 readings has three). Under the main reading (O9 (a, a, a), O10 (a), CV (a)) alpha is 1.481 +/- 0.053, beta 0.760 +/- 0.056, mu 1.575 +/- 0.181 (44 sessions have a window with r^2 >= 0.85) and z 0.156 +/- 0.043: beta is within tolerance, alpha is above 1.41 +/- 0.05, mu is about twice the paper's value and z about half. Beta is within tolerance under 46 readings, alpha under 24, mu and z under none. The surrogate windows give alpha 1.068 and beta 0.968. Fitting mu over ranks 1 to K/2 only (variants (c) and (d), outside O9, asked for by the verifier), 99.9% and 100% of windows pass r^2 >= 0.85 and every session has a mu, but mu is 1.06 +/- 0.07 in the active phase, still above 0.73 +/- 0.08; with these variants at most two means are within tolerance (alpha and beta, with the K range (b)), so the outcome does not change.

## fig4-exponents-passive

> In the passive phase the exponents are alpha 1.37 +/- 0.04, beta 0.78 +/- 0.04, mu 0.61 +/- 0.05 and z 0.36 +/- 0.06, differing significantly from the active phase for alpha, z and mu but not for beta.

Criterion: exponents as in fig4-exponents-active on passive windows; Mann-Whitney U between session values, two-sided at 0.05 (O11 (a)). Reproduced when all four means are within tolerance under one reading, alpha, z and mu differ significantly with passive alpha lower, z higher and mu lower, and beta does not differ (p >= 0.05). Partially reproduced when the tests agree but one mean is out of tolerance, or the means hold and one test disagrees. Window values pooled (O11 (b)) are shown as a variant for the main reading.

In [ ]:


def mwu(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    if len(a) < 2 or len(b) < 2:
        return np.nan
    return stats.mannwhitneyu(a, b, alternative="two-sided").pvalue


MP = means_table("passive", "passive")
rows = []
for cv in CVS:
    for reading in READ9:
        for agg in "ab":
            A, P = SV[("active", cv, reading, agg)], SV[("passive", cv, reading, agg)]
            t = {e: mwu(A[e], P[e]) for e in EXP}
            sign = dict(alpha=P.alpha.mean() < A.alpha.mean(), z=P.z.mean() > A.z.mean(), mu=P.mu.mean() < A.mu.mean())
            tests_ok = (all(t[e] < 0.05 and sign[e] for e in ["alpha", "z", "mu"]) and t["beta"] >= 0.05)
            n_tests = sum(t[e] < 0.05 and sign[e] for e in ["alpha", "z", "mu"]) + (t["beta"] >= 0.05)
            rows.append(dict(cv=cv, O9="".join(reading), O10=agg, **{f"p_{e}": t[e] for e in EXP},
                             tests_agree=n_tests, tests_ok=tests_ok))
TP = pd.DataFrame(rows)
MPT = MP.merge(TP, on=["cv", "O9", "O10"])
MPT["verdict"] = np.where((MPT.within == 4) & MPT.tests_ok, "reproduced",
                          np.where(((MPT.within == 3) & MPT.tests_ok) | ((MPT.within == 4) & (MPT.tests_agree == 3)),
                                   "partially_reproduced", "not_reproduced"))
print(MPT[show + ["p_alpha", "p_beta", "p_mu", "p_z", "tests_agree", "verdict"]].round(4).to_string(index=False))
print("\nverdicts over readings:", MPT.verdict.value_counts().to_dict())
fa = FITS[(FITS.kind == "real") & (FITS.cv_a <= 5)]
print("\nvariant O11 (b), window values pooled, main reading:",
      {e: f"{mwu(fa[fa.phase == 'active'][c], fa[fa.phase == 'passive'][c]):.2e}"
       for e, c in [("alpha", "alpha_a"), ("beta", "beta_aa"), ("mu", "mu_a"), ("z", "z_a")]})

**Result: not reproduced.** All 48 readings fail. Under the main reading alpha is 1.465, beta 0.764, mu 1.426 (19 sessions) and z 0.152, with one mean (beta) within tolerance. Against the active phase (Mann-Whitney, session values): alpha is lower (p = 0.028), beta does not differ (p = 0.48), as the paper says, mu is lower (p < 0.0001), and z does not differ (p = 0.41), where the paper reports passive z higher.

## fig4-exponents-nni

> With non-natural images the exponents are alpha 1.28 +/- 0.03, beta 0.82 +/- 0.03, mu 0.38 +/- 0.04 and z 0.4 +/- 0.1, all significantly different from both natural image phases.

Criterion: exponents on NNI windows (O4 readings); Mann-Whitney U as in M8 under O11 (a), two-sided at 0.05. Reproduced when all four means are within tolerance under one reading and all eight comparisons are significant with NNI alpha and mu lower, beta and z higher than both other phases. Partially reproduced when at least six comparisons hold and at least three means are within tolerance.

In [ ]:
rows = []
for nni in ["nni_a", "nni_b"]:
    M = means_table(nni, "nni")
    M["O4"] = nni
    for i, row in M.iterrows():
        reading = tuple(row.O9)
        Nn = SV[(nni, row.cv, reading, row.O10)]
        nholds = 0
        ps = {}
        for other in ["active", "passive"]:
            O = SV[(other, row.cv, reading, row.O10)]
            for e, lower in [("alpha", True), ("mu", True), ("beta", False), ("z", False)]:
                p = mwu(Nn[e], O[e])
                ps[f"p_{e}_{other}"] = p
                direction = Nn[e].mean() < O[e].mean() if lower else Nn[e].mean() > O[e].mean()
                nholds += bool(p < 0.05 and direction)
        rows.append(dict(row.to_dict(), comparisons_hold=nholds, **ps))
MN = pd.DataFrame(rows)
MN["verdict"] = np.where((MN.within == 4) & (MN.comparisons_hold == 8), "reproduced",
                         np.where((MN.within >= 3) & (MN.comparisons_hold >= 6), "partially_reproduced", "not_reproduced"))
print(MN[["O4"] + show + ["comparisons_hold", "verdict"]].round(3).to_string(index=False))
print("\nverdicts over readings:", MN.verdict.value_counts().to_dict())
m = MN[(MN.O4 == "nni_a") & (MN.cv == "a") & (MN.O9 == "aaa") & (MN.O10 == "a")].iloc[0]
print("\nmain reading p-values:", {k: f"{v:.2e}" for k, v in m.items() if str(k).startswith("p_")})

**Result: not reproduced.** All 96 readings fail, as none puts three means within tolerance. Under the main reading (NNI (a)) alpha is 1.398, beta 0.836, mu 1.338 (7 sessions) and z 0.192; only beta is within tolerance. Seven of the eight comparisons hold in the stated direction (alpha, beta and z against both phases, mu against active; p from 3.3e-4 to 6.5e-20); mu against passive does not (p = 0.74).

## fig4bc-static-exponents-trend

> In the active phase alpha and beta deviate significantly from their trivial values of 1; in the passive phase they lie closer to the trivial values, and closer still with non-natural images, while z moves away from its trivial value 0 from active to passive to non-natural images.

Criterion: the deviation test is (a) session alpha and beta of the active phase against those of the surrogate windows, Mann-Whitney U, or (b) a one-sample two-sided Wilcoxon signed-rank test against 1. Reproduced when, under one reading, active alpha and beta differ significantly (p < 0.05) from trivial with alpha above 1 and beta below 1, and the mean distances |alpha - 1|, |1 - beta| and z are ordered active > passive > NNI for alpha, active >= passive > NNI for beta and active < passive < NNI for z. Partially reproduced when the deviation holds and one ordering fails.

In [ ]:
rows = []
for cv in CVS:
    for reading in READ9:
        for agg in "ab":
            A = SV[("active", cv, reading, agg)]
            Su = session_values("active", cv, reading, agg, kind="surrogate") if (agg == "a") else None
            P = SV[("passive", cv, reading, agg)]
            for nni in ["nni_a", "nni_b"]:
                Nn = SV[(nni, cv, reading, agg)]
                da = [abs(x.alpha.mean() - 1) for x in (A, P, Nn)]
                db = [abs(1 - x.beta.mean()) for x in (A, P, Nn)]
                dz = [x.z.mean() for x in (A, P, Nn)]
                o_alpha = da[0] > da[1] > da[2]
                o_beta = db[0] >= db[1] > db[2]
                o_z = dz[0] < dz[1] < dz[2]
                p_w = (stats.wilcoxon(A.alpha.dropna() - 1).pvalue, stats.wilcoxon(A.beta.dropna() - 1).pvalue)
                p_s = (mwu(A.alpha, Su.alpha), mwu(A.beta, Su.beta)) if Su is not None else (np.nan, np.nan)
                dev_w = p_w[0] < .05 and p_w[1] < .05 and A.alpha.mean() > 1 and A.beta.mean() < 1
                dev_s = (p_s[0] < .05 and p_s[1] < .05 and A.alpha.mean() > Su.alpha.mean() and A.beta.mean() < Su.beta.mean()
                         and A.alpha.mean() > 1 and A.beta.mean() < 1) if Su is not None else False
                nord = int(o_alpha) + int(o_beta) + int(o_z)
                for test, dev, ps in [("a surrogate", dev_s, p_s), ("b Wilcoxon vs 1", dev_w, p_w)]:
                    if test.startswith("a") and Su is None:
                        continue
                    v = "reproduced" if dev and nord == 3 else "partially_reproduced" if dev and nord == 2 else "not_reproduced"
                    rows.append(dict(cv=cv, O9="".join(reading), O10=agg, O4=nni, test=test, p_alpha=ps[0], p_beta=ps[1],
                                     deviation=dev, d_alpha=np.round(da, 3).tolist(), d_beta=np.round(db, 3).tolist(),
                                     z=np.round(dz, 3).tolist(), order_alpha=o_alpha, order_beta=o_beta, order_z=o_z,
                                     verdict=v))
BC = pd.DataFrame(rows)
print(BC[(BC.cv == "a") & (BC.O4 == "nni_a")].to_string(index=False))
print("\nverdicts over all readings:", BC.verdict.value_counts().to_dict())
# Verifier round 1: the surrogate comparison paired by session (Wilcoxon signed-rank), main reading
A_ = SV[("active", "a", ("a", "a", "a"), "a")]
Su_ = session_values("active", "a", ("a", "a", "a"), "a", kind="surrogate")
j = A_.merge(Su_, left_index=True, right_index=True, suffixes=("", "_surr"))
print(f"paired by session ({len(j)} sessions): alpha active - surrogate {np.mean(j.alpha - j.alpha_surr):.3f},",
      f"p = {stats.wilcoxon(j.alpha - j.alpha_surr).pvalue:.1e}; beta {np.mean(j.beta - j.beta_surr):.3f},",
      f"p = {stats.wilcoxon(j.beta - j.beta_surr).pvalue:.1e}")

**Result: partially reproduced.** Under every reading the deviation holds and one ordering fails. Under the main reading active alpha is above 1 and beta below 1 (Wilcoxon against 1, p = 1.2e-17 for each; against the surrogate session values, p = 2.5e-33 for each), and the distances from the trivial values are ordered active > passive > NNI for alpha (0.481, 0.465, 0.398) and for beta (0.240, 0.236, 0.164). z does not move away from 0 in the stated order: 0.156 active, 0.152 passive, 0.192 NNI. Paired by session with its surrogate (Wilcoxon signed-rank, 97 sessions), alpha exceeds the surrogate value by 0.413 and beta falls short of it by 0.208, p = 1.2e-17 for each.

## figs3-exponents-stable-with-threshold

> Exponents computed only from windows with kurtosis above the threshold vary only slightly with the threshold and stay within their error bars.

Criterion: per phase, exponents as in fig4-exponents-active, the session value over the session's windows with kappa > kappa_th (sessions with no such window left out), on the grid from kappa_surr_max to 12. Reproduced when, for every phase and exponent, the mean across sessions stays within one SD across sessions (at kappa_th = kappa_surr_max) of its all-window value, at every grid point with at least 10 sessions contributing. Partially reproduced when it holds for three of the four exponents. Main reading of O9 and O10 (a); the CV readings and O6 are run; the thresholds 4 to 12 are shown as a variant.

In [ ]:


def stable_eval(cv, g, reading=("a", "a", "a"), phases=("active", "passive", "nni_a")):
    """The criterion at every point of the grid g (session values computed once per point)."""
    rows = []
    for ph in phases:
        base = SV[(ph, cv, reading, "a")]
        per = [session_values(ph, cv, reading, "a", kmin=th) for th in g]
        for e in EXP:
            allv = base[e].mean()
            sd = per[0][e].std() if len(g) and len(per[0]) else np.nan
            dep = [abs(s_[e].mean() - allv) for s_ in per if len(s_) and s_[e].notna().sum() >= 10]
            most = max([int(s_[e].notna().sum()) for s_ in per if len(s_)] or [0])
            worst = max(dep) if dep else 0.0
            rows.append(dict(cv=cv, phase=ph, exponent=e, all_windows=allv, sd_at_lo=sd, points=len(dep),
                             most_sessions=most, largest_departure=worst,
                             within=bool(len(dep) > 0 and worst <= sd)))
    return pd.DataFrame(rows)


def stable_verdict(t):
    ok_e = t.groupby("exponent").within.all()
    return ("reproduced" if ok_e.all() else "partially_reproduced" if ok_e.sum() >= 3 else "not_reproduced"), \
        ",".join(ok_e.index[~ok_e])


TABLES3 = {}
rows = []
for cv in CVS:
    for sm in SMAXS:
        ksm = surr_max(cv, "a", sm)
        g = grid(ksm, 12)
        row = dict(cv=cv, start=f"O6 ({sm})", kappa_start=ksm, grid_points=len(g))
        if len(g):
            t = stable_eval(cv, g)
            TABLES3[(cv, f"O6 ({sm})")] = t
            print(f"cv {cv}, O6 {sm}, every point of the grid from {g[0]} to 12:")
            print(t.round(3).to_string(index=False))
            row["verdict"], row["failing"] = stable_verdict(t)
        else:
            row["verdict"] = "no grid"
        rows.append(row)
# Verifier round 1: grids started below kappa_surr_max (CV (a) and (c))
for cv in ["a", "c"]:
    for name, st in STARTS.items():
        g = grid(st, 12)
        if len(g) == 0:
            rows.append(dict(cv=cv, start=name, kappa_start=st, grid_points=0, verdict="no grid"))
            continue
        t = stable_eval(cv, g)
        TABLES3[(cv, name)] = t
        print(f"cv {cv}, start {name}, every point of the grid from {g[0]} to 12:")
        print(t.round(3).to_string(index=False))
        v, f = stable_verdict(t)
        rows.append(dict(cv=cv, start=name, kappa_start=st, grid_points=len(g), verdict=v, failing=f))
print("\nverdicts:")
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\nvariant, thresholds 4 to 12 in steps of 0.5, cv a:")
VS3 = stable_eval("a", np.arange(4, 12.01, 0.5))
TABLES3[("a", "4 to 12")] = VS3
print(VS3.round(3).to_string(index=False))

In [ ]:
# Verifier round 2: the criterion read literally. "At every grid point with at least 10 sessions
# contributing" holds vacuously for an exponent with no such point; the verdict above counts that
# exponent as not shown stable. Both conventions, for every grid run above.
rows = []
for (cv, name), t in TABLES3.items():
    v = t.assign(within=(t.points == 0) | (t.largest_departure <= t.sd_at_lo))
    lit, lit_f = stable_verdict(v)
    strict, strict_f = stable_verdict(t)
    rows.append(dict(cv=cv, start=name, verdict_no_point_fails=strict, failing=strict_f,
                     verdict_literal=lit, failing_literal=lit_f,
                     no_point=",".join(f"{r.phase}:{r.exponent}" for r in t.itertuples() if r.points == 0)))
print(pd.DataFrame(rows).to_string(index=False))
print("\nmost sessions with a mu at any grid point, start 7, cv a:")
print(TABLES3[("a", "paper 7")].query("exponent == 'mu'")[["phase", "points", "most_sessions"]].to_string(index=False))


**Result: partially reproduced**, resting on CV (a) and on the grid start of 7, the paper's value of kappa_surr_max (M4), both assumptions. The criterion's grid from kappa_surr_max (34.29) is empty under CV (a) and (c). From 7 to 12 (101 points, every point evaluated), alpha and beta stay within one SD of their all-window values in every phase, mu in the active phase (largest departure 0.122 against an SD of 0.197), and z in the active and passive phases; in NNI (a), z departs by 0.083 against an SD of 0.058. In the passive and NNI phases mu has no grid point with 10 sessions contributing (at most 7 and 6, last cell), so the criterion, which asks for stability "at every grid point with at least 10 sessions contributing", holds vacuously there (verifier, rounds 2 and 3). z alone fails, so three of four exponents hold, which is partially reproduced. The starts disagree: from the 99th percentile (6.51) the outcome is the same, from the 99.9th (10.43) and under CV (b) z also holds (reproduced; last cell, column `verdict_literal`).

Two caveats. Passive and NNI mu are not actually tested, since only 1.7% of windows pass the r^2 rule, so the outcome rests partly on a condition that holds vacuously. The NNI z failure is narrow: its departure of 0.083 is the same at every start, and whether it fails depends only on the SD at the start of the grid. The verdicts printed two cells above (and in the column `verdict_no_point_fails` of the last cell) count an exponent with no evaluable point as not shown stable, the convention of rounds 1 and 2, which the criterion does not state; under it the grid from 7 gives not reproduced.

## Summary

| claim | outcome |
|---|---|
| sessions-selected | not reproduced |
| fig3a-kurtosis-broader-than-surrogate | partially reproduced (assumed) |
| fig3c-scaling-hit-rate-higher | not reproduced (assumed) |
| fig3c-hit-rate-rises-with-threshold | not reproduced (assumed) |
| fig3d-relative-gain-24 | not reproduced |
| figs1-other-window-lengths | not reproduced |
| figs2-strict-units | not reproduced (assumed) |
| fig4a-active-most-scaling | not reproduced (assumed) |
| fig4-exponents-active | not reproduced |
| fig4-exponents-passive | not reproduced |
| fig4-exponents-nni | not reproduced |
| fig4bc-static-exponents-trend | partially reproduced |
| figs3-exponents-stable-with-threshold | partially reproduced (assumed) |
| figs4-initial-kurtosis-poor-predictor | reproduced |
| figs5a-firing-rate-order | reproduced (assumed) |
| figs5c-hit-rate-firing-rate-nonmonotonic | reproduced |
| figs7-fatigue-halves | not reproduced (assumed) |
| figs8-halves-and-phase-order | not reproduced (assumed) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The paper links no code. The kurtosis follows Eqs. 2 to 5 as written, and on these data the ISI-shuffled surrogates give kurtoses around 3.6 (minimum 2.80), where the paper reports surrogate kurtoses no lower than about 6 and a surrogate maximum of about 7; the real windows lie about 2.5 above the surrogates here, so the kurtosis scale of the paper is not reached and the fixed thresholds of the criteria (8, 12, 12.5) do not mean the same here. The surrogate maximum is set by a few windows (34.29 in the active phase), which leaves the plotted threshold ranges above it empty; four of the claims judged on those grids are judged with the grid started at every threshold below it and give the same outcome at every start; figs3-exponents-stable-with-threshold is judged only at named starts (7, the 99th and 99.9th percentiles) and on a grid from 4, its outcome differs between them, and the paper's value of about 7 is adopted. The correlation matrix in place of the covariance does not recover the paper's surrogate kurtoses either. The analysis set has 97 sessions from 68 mice, not 62 from 44. Units silent in a window are left out of that window. Bootstrap SEs use 10^4 Poisson resamples, not 10^5 resamples of fixed size. The readings of the claims file are all run; where method readings disagree, CV (a) is adopted as an assumption.

What was found about the dataset. 153 sessions from 81 mice; no session is deposited twice. 97 sessions (68 mice) have more than 128 V1 units, the `good` label gives 68 (54 mice) and the default quality filters give 1; the paper's 62 sessions (44 mice) and 43 strict sessions match no reading. The stimulus blocks are in the order active, gray screen (10 s), Gabors, gray screen (about 5 min), flashes, passive, where the Appendix puts the gray screen last. Image onsets evoke V1 responses and omitted flashes do not, in every session, so spikes and stimulus tables share a clock. Active windows hold 0 to 4 go trials (median 2), not the "three to five" of the Appendix; counting aborted trials as failures gives a median of 5. In a few windows most V1 units are silent (up to 203 units in one window).

In [ ]:
print(f'notebook run time: {(time.time() - T_START) / 60:.1f} min')